## LIBRARIES

In [ ]:
import os
import json
import time
import random
import logging
import joblib
import math
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.colors import ListedColormap

from IPython.display import display

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy import stats
from scipy.stats import norm
from scipy.interpolate import griddata
from scipy.spatial import ConvexHull
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout, Reshape, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model

import base64
from io import BytesIO

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

tf.config.experimental.enable_op_determinism()

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.FATAL)

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')


## PREPARATION

### Vessel Data Acquisition and Feature Selection


In [ ]:
input_path = r"C:\Users\siani\Desktop\DF" 
vessel_names = ["ECO ADRIATICA", "ECO LIVORNO"]
datasets = {}

# Define a robust physical interpolation filter for target fuel flows
def clean_flow_dropouts_interpolation(df, col):
    y = df[col].values.copy()
    
    # Convert any values below 0.3 mt/h to NaN (since harbor and maneuvers are already excluded)
    df_temp = pd.Series(y)
    anom_mask = y < 0.3
    df_temp[anom_mask] = np.nan
    
    # Apply linear interpolation to fill the sensor gaps and forward/backward fill the rest
    y_clean = df_temp.interpolate(method='linear').ffill().bfill().values
    return y_clean
print("loading datasets")
# Weather directional variables to transform from degrees to sin/cos
dir_columns_to_transform = ['ep_SHIP_SEADIR_1', 'ep_WH_SWELLD_1', 'ep_WH_WAVED_1', 'ep_WH_DIR_1']
for name in vessel_names:
    safe_name = name.replace(" ", "_")
    file_path = os.path.join(input_path, f"refined_15min_{safe_name}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep=',')
        df['ts'] = pd.to_datetime(df['ts'])
        df = df.sort_values('ts')
        
        # Apply the new robust physical interpolation filter
        df['ep_ME_FLW_SUP_1'] = clean_flow_dropouts_interpolation(df, 'ep_ME_FLW_SUP_1')
        df['ep_ME_FLW_SUP_2'] = clean_flow_dropouts_interpolation(df, 'ep_ME_FLW_SUP_2')
        
        # TRIGONOMETRIC TRANSFORMATION OF METEO DIRECTIONS
        
        for col in dir_columns_to_transform:
            if col in df.columns:
                # Convert degrees to radians
                rad = np.deg2rad(df[col])
                
                # Create sine and cosine columns
                df[f'{col}_SIN'] = np.sin(rad)
                df[f'{col}_COS'] = np.cos(rad)
                
                # Drop the raw degree column to avoid confusing the models
                df.drop(columns=[col], inplace=True)
                
        datasets[name] = df
        print(f"{name.lower()} loaded, physically cleaned and directions vectorized")
    else:
        print(f"error file not found for {name.lower()}")
print("operation completed datasets ready for prediction")
navigation_vars = ['ep_SHIP_SOG_1', 'ep_SHIP_STW_1', 'ep_SHIP_DRAFTAFT_1', 'ep_SHIP_DRAFTFOR_1', 'ep_SHIP_HEAD_1', 'miglia']
power_vars = ['SHA_POW_TOT', 'SHG_POW_TOT']
# Updated with SIN/COS versions, removing the raw angle variables
weather_vars = [
    'ep_SHIP_SEAF_1', 'ep_SHIP_STEMP_1', 'ep_WH_AIRT_1', 
    'ep_WH_SWELLH_1', 'ep_WH_SWELLP_1', 
    'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 
    'ep_WH_SPEED_1',
    'ep_SHIP_SEADIR_1_SIN', 'ep_SHIP_SEADIR_1_COS',
    'ep_WH_SWELLD_1_SIN', 'ep_WH_SWELLD_1_COS',
    'ep_WH_WAVED_1_SIN', 'ep_WH_WAVED_1_COS',
    'ep_WH_DIR_1_SIN', 'ep_WH_DIR_1_COS'
]
                   
position_vars   = ['ep_SHIP_LAT_1', 'ep_SHIP_LON_1']
target_vars = ['ep_ME_FLW_SUP_1', 'ep_ME_FLW_SUP_2']
required_features = navigation_vars + power_vars + weather_vars + target_vars + position_vars

### Active Telemetry Filtering and Dataset Loading


In [ ]:
def remove_sensor_faults(df):
    df_clean = df.copy()
    
    mask_dropouts = (df_clean['ep_SHIP_SOG_1'] > 8.0) & ((df_clean['ep_ME_FLW_SUP_1'] < 0.7) | (df_clean['ep_ME_FLW_SUP_2'] < 0.7))
    df_clean = df_clean[~mask_dropouts].copy()
    
    df_clean['is_stuck'] = df_clean['ep_ME_FLW_SUP_1'].diff().abs() < 1e-4
    df_temp = df_clean.copy()
    df_temp['stuck_group'] = (df_temp['is_stuck'] != df_temp['is_stuck'].shift()).cumsum()
    df_temp['stuck_duration'] = df_temp.groupby('stuck_group')['ep_ME_FLW_SUP_1'].transform('size')
    
    mask_flatlines = (
        (df_temp['stuck_duration'] >= 6) & 
        (df_temp['ep_SHIP_SOG_1'] > 5.0) & 
        (df_temp['ep_ME_FLW_SUP_1'].between(0.94, 1.05))
    )
    
    df_final = df_temp[~mask_flatlines].copy()
    df_final.drop(columns=['is_stuck', 'stuck_group', 'stuck_duration'], inplace=True, errors='ignore')
    return df_final

print("Loading and filtering datasets...")

for name in vessel_names:
    safe_name = name.replace(" ", "_")
    file_path = os.path.join(input_path, f"refined_15min_{safe_name}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep=',')
        df['ts'] = pd.to_datetime(df['ts'])
        df = df.sort_values('ts')
        
        initial_len = len(df)
        df_filtered = remove_sensor_faults(df)
        removed_rows = initial_len - len(df_filtered)
        
        datasets[name] = df_filtered
        print(f"\n{name}: {df_filtered.shape[0]} samples loaded (removed {removed_rows} telemetry sensor faults)")
    else:
        print(f" !! error: file not found for {name}")

print(f"\nOperation completed: {len(datasets)} datasets ready for prediction")


### ROUTE STATISTICAL SUMMARY

In [ ]:
for name, df in datasets.items():
    print(f"\nanalyzing data integrity for: {name}")
       
    trip_stats = df.groupby('Voyage_ID').agg(
        route=('Route', 'first'),
        ts_min=('ts', 'min'),
        ts_max=('ts', 'max'),
        obs=('ts', 'count')
    )
    
    trip_stats['duration_real_h'] = (trip_stats['ts_max'] - trip_stats['ts_min']).dt.total_seconds() / 3600
    
    summary = trip_stats.groupby('route').agg(
        total_samples=('obs', 'sum'),
        trip_count=('obs', 'count'),
        avg_samples_per_trip=('obs', 'mean'),
        total_real_duration=('duration_real_h', 'sum')
    )
    
    summary['avg_data_hours'] = (summary['avg_samples_per_trip'] * 0.25).round(1)
    summary['total_real_hours'] = summary['total_real_duration'].round(1)
    
    summary['integrity_score'] = ((summary['avg_data_hours'] / (summary['total_real_hours'] / summary['trip_count'])) * 100).round(1)
    
    summary = summary.sort_values(by='total_samples', ascending=False)
    
    print(summary[['total_samples', 'trip_count', 'total_real_hours', 'integrity_score']].to_string())

print("\ndata integrity analysis completed")


### Correlation Analysis

In [ ]:
input_vectors = {}

for name, df in datasets.items():
    print(f"\nfeature engineering and correlation analysis for {name.lower()}")
    
    # LA RETE VEDE SOLO LA PERFEZIONE DEI DATI FISICI
    # Le variabili categoriche verranno usate SOLO DOPO la previsione LSTM per l'export JSON
    
    all_cols = df.columns.tolist()
    present_features = [v for v in required_features if v in all_cols]
    
    dynamic_physical = [v for v in present_features if df[v].nunique() > 1]
    constant_features = [v for v in present_features if v not in dynamic_physical]
    
    if constant_features:
        print(f"info removed constant variables: {constant_features}")

    # Escludiamo esplicitamente 'Route' e 'Cluster_Number' (e il target) dai tensori LSTM
    optimized_input = [v for v in dynamic_physical if v not in target_vars and v not in ['Route', 'Cluster_Number']]
    
    input_vectors[name] = optimized_input
    
    print(f"model input size: {len(optimized_input)} features (PURE PHYSICAL SENSORS)")
    
    analysis_vars = [t for t in target_vars if t in dynamic_physical] + [v for v in dynamic_physical if v in optimized_input and v not in target_vars]
    
    plt.figure(figsize=(16, 10))
    # import seaborn as sns
    sns.heatmap(df[analysis_vars].corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
    plt.title(f"sensor correlation with fuel flow meters for {name.lower()}", fontweight='bold')
    plt.show()

## LSTM TENSOR PREPARATION

In [ ]:
lookback = 16          
horizon = 1           
min_trip_rows = 21 # 5 h and 15 min    
target_features = ['ep_ME_FLW_SUP_1', 'ep_ME_FLW_SUP_2']

lstm_data = {}
vessel_scalers = {}

# Global list to collect route distribution across all vessels for final pivot table
global_route_stats = []

def create_sequences_by_trip(df, features, targets, lb, hr):
    xs, ys, ts_list = [], [], []
    for _, df_trip in df.groupby('Voyage_ID'):
        if len(df_trip) >= lb + hr:
            x_vals = df_trip[features].values
            y_vals = df_trip[targets].values
            ts_vals = pd.to_datetime(df_trip['ts']).values
            
            for i in range(len(x_vals) - lb - hr + 1):
                time_diff = (ts_vals[i + lb + hr - 1] - ts_vals[i]) / np.timedelta64(1, 'm')
                expected_diff = (lb + hr - 1) * 15
                
                if time_diff == expected_diff:
                    xs.append(x_vals[i : i + lb])
                    target_seq = y_vals[i + lb : i + lb + hr]
                    ys.append(target_seq.flatten() if hr == 1 else target_seq)
                    ts_list.append(ts_vals[i + lb])
                    
    return np.array(xs), np.array(ys), np.array(ts_list)

for name, df_raw in datasets.items():
    print(f"\npreparing tensors and temporal split for: {name}")
    df = df_raw.copy().sort_values('ts')
    input_cols = [f for f in input_vectors[name] if f in df.columns and f not in target_features]
    
    trip_counts = df.groupby('Voyage_ID').size()
    valid_trip_ids = trip_counts[trip_counts >= min_trip_rows].index.tolist()
   
    # Split by route:
    # TRAIN 70% - VAL 15% - TEST 15%
    train_ids = []
    val_ids = []
    test_ids = []
    
    # Recover 'Route' column if dropped during preprocessing
    if 'Route' not in df.columns:
        for var_name, var_data in dict(globals()).items():
            if isinstance(var_data, pd.DataFrame) and 'Voyage_ID' in var_data.columns and 'Route' in var_data.columns:
                df_map = var_data.drop_duplicates(subset=['Voyage_ID'])
                mappa_temp = dict(zip(df_map['Voyage_ID'], df_map['Route']))
                df['Route'] = df['Voyage_ID'].map(mappa_temp)
                break
                
    if 'Route' not in df.columns:
        df['Route'] = "Unknown Route"
            
    unique_routes = df['Route'].dropna().unique()

    for route in unique_routes:
        route_voyages = df[(df['Route'] == route) & (df['Voyage_ID'].isin(valid_trip_ids))]['Voyage_ID'].unique()
        n_trips = len(route_voyages)
        
        if n_trips == 0: 
            continue
        elif n_trips < 3: 
            # Force rare routes (1-2 trips) into Training set to handle Out-Of-Distribution samples
            train_ids.extend(route_voyages)
            continue
            
        train_split = int(n_trips * 0.7) # 70%
        val_split = int(n_trips * 0.85) # 15%
        
        # Balance allocation to ensure at least 1 trip in Train, Val, and Test
        if train_split == 0: train_split = 1
        if val_split <= train_split: val_split = train_split + 1
        if val_split >= n_trips: 
            val_split = n_trips - 1
            if train_split >= val_split:
                train_split = val_split - 1
        
        train_ids.extend(route_voyages[:train_split])
        val_ids.extend(route_voyages[train_split:val_split])
        test_ids.extend(route_voyages[val_split:])
    
    train_df = df[df['Voyage_ID'].isin(train_ids)].copy()
    val_df = df[df['Voyage_ID'].isin(val_ids)].copy()
    test_df = df[df['Voyage_ID'].isin(test_ids)].copy()

    def get_date_range(df_set):
        if df_set.empty: return "n/a"
        start = df_set['ts'].min().strftime('%d/%m/%Y')
        end = df_set['ts'].max().strftime('%d/%m/%Y')
        return f"{start} to {end}"

    sc_x, sc_y = MinMaxScaler(), MinMaxScaler()
    
    if not train_df.empty:
        train_df[input_cols] = sc_x.fit_transform(train_df[input_cols])
        train_df[target_features] = sc_y.fit_transform(train_df[target_features])

    for d_set in [val_df, test_df]:
        if not d_set.empty:
            d_set[input_cols] = sc_x.transform(d_set[input_cols])
            d_set[target_features] = sc_y.transform(d_set[target_features])

    vessel_scalers[name] = {'scaler_x': sc_x, 'scaler_y': sc_y}

    x_train, y_train, ts_train = create_sequences_by_trip(train_df, input_cols, target_features, lookback, horizon) if not train_df.empty else (np.array([]), np.array([]), np.array([]))
    x_val, y_val, ts_val = create_sequences_by_trip(val_df, input_cols, target_features, lookback, horizon) if not val_df.empty else (np.array([]), np.array([]), np.array([]))
    x_test, y_test, ts_test = create_sequences_by_trip(test_df, input_cols, target_features, lookback, horizon) if not test_df.empty else (np.array([]), np.array([]), np.array([]))

    lstm_data[name] = {
        'x_train': x_train, 'y_train': y_train,
        'x_val': x_val, 'y_val': y_val,
        'x_test': x_test, 'y_test': y_test,
        'features_name': input_cols,
        'ts_test': ts_test}

    stats_data = [
        ["training", x_train.shape[0] if x_train.size > 0 else 0, get_date_range(train_df), len(train_ids)],
        ["validation", x_val.shape[0] if x_val.size > 0 else 0, get_date_range(val_df), len(val_ids)],
        ["test", x_test.shape[0] if x_test.size > 0 else 0, get_date_range(test_df), len(test_ids)]]
    
    summary_table = pd.DataFrame(stats_data, columns=["set", "sequences", "time period", "voyage count"])
    print(summary_table.to_string(index=False))
    print("-" * 55)
    
    # Collect route statistics for the final pivot table
    for route in unique_routes:
        route_v = df[df['Route'] == route]['Voyage_ID'].unique()
        tr = len([v for v in route_v if v in train_ids])
        vl = len([v for v in route_v if v in val_ids])
        ts = len([v for v in route_v if v in test_ids])
        
        if (tr + vl + ts) > 0:
            global_route_stats.append({'Route': route, 'Vessel': name, 'Train': tr, 'Val': vl, 'Test': ts})


#Pivot Table
print(" GLOBAL ROUTE DISTRIBUTION SUMMARY (ALL VESSELS) ")

if global_route_stats:
    df_global = pd.DataFrame(global_route_stats)
    
    vessels = sorted(df_global['Vessel'].unique())
    unique_routes_all = df_global['Route'].unique()
    
    # Construct flat pivot format
    pivot_data = []
    for route in unique_routes_all:
        row_dict = {'Route': route}
        for v in vessels:
            subset = df_global[(df_global['Route'] == route) & (df_global['Vessel'] == v)]
            if not subset.empty:
                row_dict[f"{v} (Tr)"] = int(subset['Train'].iloc[0])
                row_dict[f"{v} (Val)"] = int(subset['Val'].iloc[0])
                row_dict[f"{v} (Ts)"] = int(subset['Test'].iloc[0])
            else:
                row_dict[f"{v} (Tr)"] = 0
                row_dict[f"{v} (Val)"] = 0
                row_dict[f"{v} (Ts)"] = 0
        pivot_data.append(row_dict)
        
    df_pivot = pd.DataFrame(pivot_data)
    
    # Sort by total volume of voyages across all vessels to prioritize active routes
    df_pivot['Total_Vol'] = df_pivot.drop(columns=['Route']).sum(axis=1)
    df_pivot = df_pivot.sort_values('Total_Vol', ascending=False).drop(columns=['Total_Vol'])
    
    print(df_pivot.to_string(index=False))
else:
    print("No valid routes found.")
print("===============================================================================================\n")


### Post-filter route check

In [ ]:
for name, df_raw in datasets.items():
    print(f"vessel: {name}")

    df = df_raw.copy().sort_values('ts')

    if 'Route' not in df.columns:
        for var_name, var_data in dict(globals()).items():
            if isinstance(var_data, pd.DataFrame) and 'Voyage_ID' in var_data.columns and 'Route' in var_data.columns:
                df_map = var_data.drop_duplicates(subset=['Voyage_ID'])
                mappa_temp = dict(zip(df_map['Voyage_ID'], df_map['Route']))
                df['Route'] = df['Voyage_ID'].map(mappa_temp)
                break
    if 'Route' not in df.columns:
        df['Route'] = "Unknown Route"

    trip_counts = df.groupby('Voyage_ID').size()
    valid_trip_ids = trip_counts[trip_counts >= min_trip_rows].index.tolist()
    df_filtered = df[df['Voyage_ID'].isin(valid_trip_ids)]

    trip_stats = df_filtered.groupby('Voyage_ID').agg(
        route=('Route', 'first'),
        ts_min=('ts', 'min'),
        ts_max=('ts', 'max'),
        obs=('ts', 'count')
    )
    trip_stats['duration_real_h'] = (trip_stats['ts_max'] - trip_stats['ts_min']).dt.total_seconds() / 3600

    summary = trip_stats.groupby('route').agg(
        total_samples=('obs', 'sum'),
        trip_count=('obs', 'count'),
        avg_samples_per_trip=('obs', 'mean'),
        total_real_hours=('duration_real_h', 'sum')
    )

    summary['avg_voyage_h']   = (summary['avg_samples_per_trip'] * 0.25).round(1)
    summary['total_real_hours'] = summary['total_real_hours'].round(1)
    summary = summary.sort_values(by='total_samples', ascending=False)

    print(summary[['total_samples', 'trip_count', 'avg_voyage_h', 'total_real_hours']].to_string())
    print("-" * 65)

print("\npost-filter integrity check completed")


### Data Preparation Summary

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import base64
from io import BytesIO

def fig_to_base64(fig):
    tmpfile = BytesIO()
    fig.savefig(tmpfile, format='png', bbox_inches='tight', facecolor='#161b22', edgecolor='none')
    plt.close(fig)
    return base64.b64encode(tmpfile.getvalue()).decode()

if not lstm_data:
    print("!! Error: lstm_data is empty. Please run the data preparation block first.")
else:
    tab_buttons = ""
    tab_contents = ""
    first_tab = True

    pearson_text = """
    <p style="color: #8b949e; line-height: 1.6; font-size: 0.95em;">
    <b>Pearson Correlation Coefficient:</b> The Pearson correlation coefficient measures the linear relationship between continuous variables (-1 to +1). In this matrix, <b>all categorical route variables have been intentionally excluded</b> to focus purely on the physical ship telemetry and weather parameters influencing main engine fuel consumption.
    </p>
    <div style="background-color: #1c2128; padding: 15px; border-radius: 6px; margin: 15px 0; text-align: center; color: #58a6ff; font-family: 'Courier New', Courier, monospace; font-weight: bold; font-size: 1.1em;">
        r = &Sigma; (x<sub>i</sub> - x&#772;)(y<sub>i</sub> - y&#772;) / &radic;[ &Sigma; (x<sub>i</sub> - x&#772;)&sup2; &Sigma; (y<sub>i</sub> - y&#772;)&sup2; ]
    </div>
    """

    for name, df_raw in datasets.items():
        if name not in lstm_data: continue
        
        safe_id = name.replace(" ", "_")
        active_class = "active" if first_tab else ""
        tab_buttons += f"<button class='tablinks {active_class}' onclick=\"openVessel(event, '{safe_id}')\">{name}</button>\n"
        
        df = df_raw.copy().sort_values('ts')
        d = lstm_data[name]
        
        trip_counts = df.groupby('Voyage_ID').size()
        valid_trip_ids = trip_counts[trip_counts >= min_trip_rows].index.tolist()
        
        if 'Route' not in df.columns:
            for var_name, var_data in list(globals().items()):
                if isinstance(var_data, pd.DataFrame) and 'Voyage_ID' in var_data.columns and 'Route' in var_data.columns:
                    df_map = var_data.drop_duplicates(subset=['Voyage_ID'])
                    mappa_temp = dict(zip(df_map['Voyage_ID'], df_map['Route']))
                    df['Route'] = df['Voyage_ID'].map(mappa_temp)
                    break
                    
        if 'Route' not in df.columns: df['Route'] = "Unknown Route"
        
        unique_routes = df['Route'].dropna().unique()
        train_ids, val_ids, test_ids = [], [], []
        route_stats_data = []
        
        for route in unique_routes:
            route_voyages = df[(df['Route'] == route) & (df['Voyage_ID'].isin(valid_trip_ids))]['Voyage_ID'].unique()
            n_trips = len(route_voyages)
            
            if n_trips == 0: continue
            elif n_trips < 3:
                train_ids.extend(route_voyages)
                route_stats_data.append([route, n_trips, 0, 0])
                continue
                
            train_split = int(n_trips * 0.7)
            val_split = int(n_trips * 0.8)
            
            if train_split == 0: train_split = 1
            if val_split <= train_split: val_split = train_split + 1
            if val_split >= n_trips: 
                val_split = n_trips - 1
                if train_split >= val_split: train_split = val_split - 1
                    
            train_ids.extend(route_voyages[:train_split])
            val_ids.extend(route_voyages[train_split:val_split])
            test_ids.extend(route_voyages[val_split:])
            
            route_stats_data.append([route, len(route_voyages[:train_split]), len(route_voyages[train_split:val_split]), len(route_voyages[val_split:])])
            
        train_df = df[df['Voyage_ID'].isin(train_ids)]
        val_df = df[df['Voyage_ID'].isin(val_ids)]
        test_df = df[df['Voyage_ID'].isin(test_ids)]
        
        def get_dates(d_set):
            if d_set.empty: return "N/A"
            return f"{d_set['ts'].min().strftime('%d/%m/%Y')} to {d_set['ts'].max().strftime('%d/%m/%Y')}"
            
        total_initial_rows = len(df_raw)
        section1 = f"""
        <h3>1. Dataset Splitting & Dimensions</h3>
        <p style="color:#c9d1d9;"><b>Total Raw Datapoints Loaded:</b> {total_initial_rows:,}</p>
        <table class='stats-table'>
            <tr><th>Set</th><th>Percentage</th><th>LSTM Sequences</th><th>Historical Date Range</th></tr>
            <tr><td><b>Training</b></td><td>70%</td><td>{d['x_train'].shape[0]:,}</td><td>{get_dates(train_df)}</td></tr>
            <tr><td><b>Validation</b></td><td>10%</td><td>{d['x_val'].shape[0]:,}</td><td>{get_dates(val_df)}</td></tr>
            <tr><td><b>Test</b></td><td>20%</td><td>{d['x_test'].shape[0]:,}</td><td>{get_dates(test_df)}</td></tr>
        </table>
        """
        
        route_stats_data.sort(key=lambda x: x[1], reverse=True) 
        route_trs = "".join([f"<tr><td>{r[0]}</td><td>{r[1]}</td><td>{r[2]}</td><td>{r[3]}</td></tr>" for r in route_stats_data])
        section2 = f"""
        <h3>2. Stratified Route Distribution</h3>
        <p style="color:#8b949e; font-size: 0.9em;">Table displays the number of individual voyages allocated to each set. Rare routes are forced into the Training set.</p>
        <table class='stats-table'>
            <tr><th>Route</th><th>Train (Voyages)</th><th>Validation (Voyages)</th><th>Test (Voyages)</th></tr>
            {route_trs}
        </table>
        """
        
        cols_for_corr = [c for c in d['features_name'] if not c.startswith('Route_')] + target_features
        df_corr = df[cols_for_corr].corr(method='pearson')
        
        main_target = target_features[0]
        if main_target in df_corr.columns:
            correlations = df_corr[main_target].drop(target_features).abs().sort_values(ascending=False).head(10)
            pearson_trs = "".join([f"<tr><td>{idx}</td><td>{df_corr.loc[idx, main_target]:.3f}</td></tr>" for idx in correlations.index])
            pearson_table = f"""
            <table class='stats-table' style='width:100%;'>
                <tr><th>Top 10 Physical Sensors</th><th>Pearson (r) vs {main_target}</th></tr>
                {pearson_trs}
            </table>
            """
        else:
            pearson_table = "<p>Target not found for correlation.</p>"
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(df_corr, annot=True, fmt=".2f", cmap='mako', ax=ax, cbar=True, annot_kws={"size": 8})
        ax.set_title(f'Pearson Matrix (Excluding Routes) - {name}', color='white', pad=15)
        ax.tick_params(colors='white', labelsize=8)
        cbar = ax.collections[0].colorbar
        cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white')
        
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        plt.setp(ax.get_yticklabels(), rotation=0)
        
        b64_img = fig_to_base64(fig)
        
        section3 = f"""
        <h3>3. Feature Correlation Analysis</h3>
        {pearson_text}
        
        <div style="display: flex; gap: 20px; align-items: flex-start; margin-top:20px;">
            <div style="flex: 1;">
                {pearson_table}
            </div>
            <div style="flex: 2; text-align:center;">
                <img src="data:image/png;base64,{b64_img}" style="width:100%; border: 1px solid #30363d; border-radius:8px; box-shadow: 0 4px 8px rgba(0,0,0,0.5);">
            </div>
        </div>
        """
        
        display_style = "block" if first_tab else "none"
        tab_contents += f"""
        <div id='{safe_id}' class='tabcontent' style='display: {display_style};'>
            {section1}
            {section2}
            {section3}
        </div>
        """
        first_tab = False

    final_html = f"""
    <!DOCTYPE html>
    <html lang='en'>
    <head>
        <meta charset='UTF-8'>
        <title>Data Preparation Summary</title>
        <style>
            body {{ font-family: 'Segoe UI', 'Inter', sans-serif; background-color: #0d1117; color: #c9d1d9; padding: 40px; margin: 0; }}
            .container {{ max-width: 1200px; margin: auto; background-color: #161b22; padding: 40px; border-radius: 12px; border: 1px solid #30363d; box-shadow: 0 10px 20px rgba(0,0,0,0.6); }}
            h1 {{ color: #58a6ff; text-align: center; font-weight: 800; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 30px; }}
            h3 {{ color: #bb86fc; margin-top: 40px; font-weight: 600; border-bottom: 1px solid #30363d; padding-bottom: 10px; }}
            .stats-table {{ width: 100%; border-collapse: collapse; margin: 20px 0; background-color: #0d1117; border-radius: 6px; overflow: hidden; }}
            .stats-table th, .stats-table td {{ border: 1px solid #30363d; padding: 12px 15px; text-align: left; font-size: 0.95em; }}
            .stats-table th {{ background-color: #1f2937; color: #58a6ff; font-weight: 700; text-transform: uppercase; font-size: 0.85em; }}
            .stats-table tr:hover {{ background-color: #1c2128; }}
            .tab {{ overflow: hidden; border-bottom: 2px solid #30363d; margin-bottom: 20px; display: flex; justify-content: center; }}
            .tab button {{ background-color: #0d1117; color: #8b949e; float: left; border: none; outline: none; cursor: pointer; padding: 14px 30px; transition: 0.3s; font-size: 1.1em; font-weight: bold; border-top-left-radius: 8px; border-top-right-radius: 8px; margin: 0 5px; }}
            .tab button:hover {{ background-color: #1c2128; color: #c9d1d9; }}
            .tab button.active {{ background-color: #1f2937; color: #58a6ff; border-bottom: 3px solid #58a6ff; }}
            .tabcontent {{ padding: 20px 0; animation: fadeEffect 0.5s; }}
            @keyframes fadeEffect {{ from {{opacity: 0;}} to {{opacity: 1;}} }}
        </style>
        <script>
            function openVessel(evt, vesselName) {{
                var i, tabcontent, tablinks;
                tabcontent = document.getElementsByClassName("tabcontent");
                for (i = 0; i < tabcontent.length; i++) {{ tabcontent[i].style.display = "none"; }}
                tablinks = document.getElementsByClassName("tablinks");
                for (i = 0; i < tablinks.length; i++) {{ tablinks[i].className = tablinks[i].className.replace(" active", ""); }}
                document.getElementById(vesselName).style.display = "block";
                evt.currentTarget.className += " active";
            }}
        </script>
    </head>
    <body>
        <div class='container'>
            <h1>Data Preparation Summary</h1>
            <div class="tab">
                {tab_buttons}
            </div>
            {tab_contents}
        </div>
    </body>
    </html>
    """
    
    report_path = "Data_Preparation_Summary.html"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(final_html)
        
    print(f"Report saved to: {report_path}")


## HYPERPARAMETER OPTIMIZATION: IMPROVED GREY WOLF OPTIMIZER

### ECO Livorno

In [ ]:
def fitness_function(params, x_train_opt, y_train_opt):
    u, lr, bs, dr = int(params[0]), params[1], int(params[2]), params[3]
    n_steps, n_feat = x_train_opt.shape[1], x_train_opt.shape[2]
    n_out = y_train_opt.shape[1]
    start_time = time.time()
    
    # default activation tanh is implicitly used to enforce cudnn execution
    model = Sequential([
        Input(shape=(n_steps, n_feat)),
        LSTM(u, return_sequences=True),
        Dropout(dr),
        LSTM(int(u/2)),
        Dropout(dr),
        Dense(16, activation='relu'),
        Dense(n_out, activation='linear')
    ])
    
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    
    # surrogate proxy strategy: 10 epochs max, aggressive patience=2
    stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    
    hist = model.fit(
        x_train_opt, y_train_opt, 
        validation_split=0.2, 
        epochs=10, 
        batch_size=bs, 
        callbacks=[stop], 
        verbose=0
    )
    
    val_loss = min(hist.history['val_loss'])
    comp_efficiency = (time.time() - start_time) / 100 
    mem_usage = model.count_params() / 1000000 
    
    # multi-objective penalty aligned with the ga strategy
    total_fitness = val_loss + (0.1 * comp_efficiency) + (0.1 * mem_usage)
    K.clear_session() 
    return total_fitness

class IGWO:
    def __init__(self, n_wolves=10, max_iter=15, patience=5):
        self.n_wolves, self.max_iter, self.patience = n_wolves, max_iter, patience
        # search space boundaries: [units, learning_rate, batch_size, dropout]
        self.lb = np.array([64, 0.0005, 16, 0.1])
        self.ub = np.array([256, 0.005, 64, 0.4])
    
    def run(self, x_tr, y_tr):
        pos = np.random.uniform(self.lb, self.ub, (self.n_wolves, 4))
        alpha_pos, alpha_score = np.zeros(4), float("inf")
        beta_pos, beta_score = np.zeros(4), float("inf")
        delta_pos, delta_score = np.zeros(4), float("inf")
        stagnant_count = 0

        for t in range(self.max_iter):
            old_best_score = alpha_score
            for i in range(self.n_wolves):
                pos[i] = np.clip(pos[i], self.lb, self.ub)
                
                # print triggers instantly at the start of each individual wolf evaluation loop
                print(f"iter {t+1}/{self.max_iter} wolf {i+1}/{self.n_wolves}", flush=True)
                fit = fitness_function(pos[i], x_tr, y_tr)
                
                if fit < alpha_score: alpha_score, alpha_pos = fit, pos[i].copy()
                elif fit < beta_score: beta_score, beta_pos = fit, pos[i].copy()
                elif fit < delta_score: delta_score, delta_pos = fit, pos[i].copy()
            
            print(f" > iteration {t+1}/{self.max_iter} - best fitness: {alpha_score:.6f}", flush=True)
            
            # convergence monitoring and early stopping for optimization
            if t > 0:
                if (old_best_score - alpha_score) < 1e-6: stagnant_count += 1
                else: stagnant_count = 0
                if stagnant_count >= self.patience:
                    print(f"\n early stopping: convergence reached at iteration {t+1}", flush=True)
                    break

            # non-linear convergence factor 'a'
            a_val = 2 * (1 - (t**2 / self.max_iter**2))
            eps = 1e-10
            
            # weighted position updating based on leader performance
            w1, w2, w3 = 1.0/(alpha_score+eps), 1.0/(beta_score+eps), 1.0/(delta_score+eps)
            w_sum = w1 + w2 + w3
            
            for i in range(self.n_wolves):
                for j in range(4):
                    r1, r2 = np.random.random(), np.random.random()
                    x1 = alpha_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * alpha_pos[j] - pos[i, j])
                    r1, r2 = np.random.random(), np.random.random()
                    x2 = beta_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * beta_pos[j] - pos[i, j])
                    r1, r2 = np.random.random(), np.random.random()
                    x3 = delta_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * delta_pos[j] - pos[i, j])
                    pos[i, j] = (w1 * x1 + w2 * x2 + w3 * x3) / w_sum
                    
        return alpha_pos

# igwo configuration targeting specific vessel
TARGET_VESSEL   = "ECO ADRIATICA"
max_iter_igwo   = 10    
n_wolves        = 10    
sampling_ratio  = 1.0 

name        = TARGET_VESSEL
safe_name   = name.replace(' ', '_')
output_path = f"opt_params_{safe_name}.json" # matches filename pattern in final audit script

       
# isolation of training voyages for tuning subsets
all_trips = datasets[name]['Voyage_ID'].unique()
train_split_idx = int(len(all_trips) * 0.7)
train_ids = all_trips[:train_split_idx]
    
# sampling training data to optimize tuning execution time
n_sample = int(len(train_ids) * sampling_ratio)
sample_trips = train_ids[:n_sample]
    
df_opt = datasets[name][datasets[name]['Voyage_ID'].isin(sample_trips)].copy()
print(f"tuning performed on {len(sample_trips)} training voyages")
    
# prepare data tensors for optimization
sc_x, sc_y = vessel_scalers[name]['scaler_x'], vessel_scalers[name]['scaler_y']
feats = lstm_data[name]['features_name']
    
# data is scaled using the previously fitted scalers to maintain consistency
df_opt[feats] = sc_x.transform(df_opt[feats])
df_opt[target_features] = sc_y.transform(df_opt[target_features])

# ====== RIGA CORRETTA QUI SOTTO ======
x_opt, y_opt, _ = create_sequences_by_trip(df_opt, feats, target_features, lookback, horizon)
# =====================================

x_opt, y_opt = x_opt.astype('float32'), y_opt.astype('float32')
    
if len(x_opt) == 0:
    print("Execution bypassed: sequence tensor is empty.")
else:
    optimizer = IGWO(n_wolves=n_wolves, max_iter=max_iter_igwo)
    best_params = optimizer.run(x_opt, y_opt)
        
    # serialize optimal parameters for model persistence
    final_params = {
        "vessel_name": name, 
        "lstm_units": int(best_params[0]),
        "learning_rate": float(best_params[1]), 
        "batch_size": int(best_params[2]),
        "dropout_rate": float(best_params[3]), 
        "lookback": lookback, 
        "features_count": len(feats)
        }
        
    with open(output_path, 'w') as f:
        json.dump(final_params, f, indent=4)
        
    print(f"Tuning successful: units={final_params['lstm_units']}, lr={final_params['learning_rate']:.5f}")


### ECO Adriatica

In [ ]:
def fitness_function(params, x_train_opt, y_train_opt):
    u, lr, bs, dr = int(params[0]), params[1], int(params[2]), params[3]
    n_steps, n_feat = x_train_opt.shape[1], x_train_opt.shape[2]
    n_out = y_train_opt.shape[1]
    start_time = time.time()
    
    # default activation tanh is implicitly used to enforce cudnn execution
    model = Sequential([
        Input(shape=(n_steps, n_feat)),
        LSTM(u, return_sequences=True),
        Dropout(dr),
        LSTM(int(u/2)),
        Dropout(dr),
        Dense(16, activation='relu'),
        Dense(n_out, activation='linear')
    ])
    
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    
    # surrogate proxy strategy: 10 epochs max, aggressive patience=2
    stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    
    hist = model.fit(
        x_train_opt, y_train_opt, 
        validation_split=0.2, 
        epochs=10, 
        batch_size=bs, 
        callbacks=[stop], 
        verbose=0
    )
    
    val_loss = min(hist.history['val_loss'])
    comp_efficiency = (time.time() - start_time) / 100 
    mem_usage = model.count_params() / 1000000 
    
    # multi-objective penalty aligned with the ga strategy
    total_fitness = val_loss + (0.1 * comp_efficiency) + (0.1 * mem_usage)
    K.clear_session() 
    return total_fitness

class IGWO:
    def __init__(self, n_wolves=10, max_iter=15, patience=5):
        self.n_wolves, self.max_iter, self.patience = n_wolves, max_iter, patience
        # search space boundaries: [units, learning_rate, batch_size, dropout]
        self.lb = np.array([64, 0.0005, 16, 0.1])
        self.ub = np.array([256, 0.005, 64, 0.4])
    
    def run(self, x_tr, y_tr):
        pos = np.random.uniform(self.lb, self.ub, (self.n_wolves, 4))
        alpha_pos, alpha_score = np.zeros(4), float("inf")
        beta_pos, beta_score = np.zeros(4), float("inf")
        delta_pos, delta_score = np.zeros(4), float("inf")
        stagnant_count = 0

        for t in range(self.max_iter):
            old_best_score = alpha_score
            for i in range(self.n_wolves):
                pos[i] = np.clip(pos[i], self.lb, self.ub)
                
                # print triggers instantly at the start of each individual wolf evaluation loop
                print(f"iter {t+1}/{self.max_iter} wolf {i+1}/{self.n_wolves}", flush=True)
                fit = fitness_function(pos[i], x_tr, y_tr)
                
                if fit < alpha_score: alpha_score, alpha_pos = fit, pos[i].copy()
                elif fit < beta_score: beta_score, beta_pos = fit, pos[i].copy()
                elif fit < delta_score: delta_score, delta_pos = fit, pos[i].copy()
            
            print(f" > iteration {t+1}/{self.max_iter} - best fitness: {alpha_score:.6f}", flush=True)
            
            # convergence monitoring and early stopping for optimization
            if t > 0:
                if (old_best_score - alpha_score) < 1e-6: stagnant_count += 1
                else: stagnant_count = 0
                if stagnant_count >= self.patience:
                    print(f"\n early stopping: convergence reached at iteration {t+1}", flush=True)
                    break

            # non-linear convergence factor 'a'
            a_val = 2 * (1 - (t**2 / self.max_iter**2))
            eps = 1e-10
            
            # weighted position updating based on leader performance
            w1, w2, w3 = 1.0/(alpha_score+eps), 1.0/(beta_score+eps), 1.0/(delta_score+eps)
            w_sum = w1 + w2 + w3
            
            for i in range(self.n_wolves):
                for j in range(4):
                    r1, r2 = np.random.random(), np.random.random()
                    x1 = alpha_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * alpha_pos[j] - pos[i, j])
                    r1, r2 = np.random.random(), np.random.random()
                    x2 = beta_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * beta_pos[j] - pos[i, j])
                    r1, r2 = np.random.random(), np.random.random()
                    x3 = delta_pos[j] - (2*a_val*r1 - a_val) * abs(2*r2 * delta_pos[j] - pos[i, j])
                    pos[i, j] = (w1 * x1 + w2 * x2 + w3 * x3) / w_sum
                    
        return alpha_pos

# igwo configuration targeting specific vessel
TARGET_VESSEL   = "ECO LIVORNO"
max_iter_igwo   = 10    
n_wolves        = 10    
sampling_ratio  = 0.5 

name        = TARGET_VESSEL
safe_name   = name.replace(' ', '_')
output_path = f"opt_params_{safe_name}.json" # matches filename pattern in final audit script

       
# isolation of training voyages for tuning subsets
all_trips = datasets[name]['Voyage_ID'].unique()
train_split_idx = int(len(all_trips) * 0.7)
train_ids = all_trips[:train_split_idx]
    
# sampling training data to optimize tuning execution time
n_sample = int(len(train_ids) * sampling_ratio)
sample_trips = train_ids[:n_sample]
    
df_opt = datasets[name][datasets[name]['Voyage_ID'].isin(sample_trips)].copy()
print(f"tuning performed on {len(sample_trips)} training voyages")
    
# prepare data tensors for optimization
sc_x, sc_y = vessel_scalers[name]['scaler_x'], vessel_scalers[name]['scaler_y']
feats = lstm_data[name]['features_name']
    
# data is scaled using the previously fitted scalers to maintain consistency
df_opt[feats] = sc_x.transform(df_opt[feats])
df_opt[target_features] = sc_y.transform(df_opt[target_features])

# ====== MODIFICA APPLICATA QUI SOTTO ======
x_opt, y_opt, _ = create_sequences_by_trip(df_opt, feats, target_features, lookback, horizon)
# ==========================================

x_opt, y_opt = x_opt.astype('float32'), y_opt.astype('float32')
    
if len(x_opt) == 0:
    print("Execution bypassed: sequence tensor is empty.")
else:
    optimizer = IGWO(n_wolves=n_wolves, max_iter=max_iter_igwo)
    best_params = optimizer.run(x_opt, y_opt)
        
    # serialize optimal parameters for model persistence
    final_params = {
        "vessel_name": name, 
        "lstm_units": int(best_params[0]),
        "learning_rate": float(best_params[1]), 
        "batch_size": int(best_params[2]),
        "dropout_rate": float(best_params[3]), 
        "lookback": lookback, 
        "features_count": len(feats)
        }
        
    with open(output_path, 'w') as f:
        json.dump(final_params, f, indent=4)
        
    print(f"Tuning successful: units={final_params['lstm_units']}, lr={final_params['learning_rate']:.5f}")


## LAYER LSTM

### ECO Adriatica

In [ ]:
TARGET_VESSEL = "ECO ADRIATICA"

if 'results_storage_igwo' not in globals():
    results_storage_igwo = {}

for name in [TARGET_VESSEL]:
    if name not in lstm_data: 
        print(f"warning: {name} is not present in lstm_data skipping computation")
        continue
    
    print(f"\nbuilding and training igwo optimized model for {name.lower()}")
    
    d = lstm_data[name]
    x_tr, y_tr = d['x_train'], d['y_train']
    x_vl, y_vl = d['x_val'], d['y_val']
    x_ts, y_ts = d['x_test'], d['y_test']
    
    n_outputs = y_tr.shape[1] 
    
    param_file = f"opt_params_{name.replace(' ', '_')}.json"
    
    if os.path.exists(param_file):
        with open(param_file, "r") as f:
            p = json.load(f)
        u, lr, bs, dr = p["lstm_units"], p["learning_rate"], p["batch_size"], p["dropout_rate"]
        print(f"optimal parameters loaded from {param_file}: units={u}, lr={lr:.5f}, batch={bs}, dropout={dr:.2f}")
    else:
        print(f"optimization parameters not found at {param_file} using default configuration")
        u, lr, bs, dr = 64, 0.001, 32, 0.2

    model = Sequential([
        Input(shape=(x_tr.shape[1], x_tr.shape[2])),
        LSTM(u, return_sequences=True, activation='tanh'),
        Dropout(dr),
        LSTM(int(u/2), activation='tanh'),
        Dropout(dr),
        Dense(16, activation='relu'),
        Dense(n_outputs, activation='linear')
    ])

    model.compile(optimizer=Adam(learning_rate=lr), loss='huber', metrics=['mae'])

    print(f"starting igwo training for {name.lower()} on {len(x_tr)} sequences")
    stop = EarlyStopping(monitor='val_loss', patience=55, restore_best_weights=True)

    history = model.fit(
        x_tr, y_tr,
        validation_data=(x_vl, y_vl),
        epochs=100,
        batch_size=bs,
        callbacks=[stop],
        verbose=1
    )

    
    pred_scaled = model.predict(x_ts)
    scaler_y = vessel_scalers[name]['scaler_x'] if 'scaler_y' not in vessel_scalers[name] else vessel_scalers[name]['scaler_y']
    
    pred_real = scaler_y.inverse_transform(pred_scaled)
    real_real = scaler_y.inverse_transform(y_ts)
    

    results_storage_igwo[name] = {
        'real': real_real, 
        'pred': pred_real,
        'history': history.history
    }
    
    model_save_path = f"model_{name.replace(' ', '_')}_igwo.keras"
    model.save(model_save_path)
    print(f"model successfully saved to {model_save_path}")

performance_report = []
for name in [TARGET_VESSEL]:
    if name not in results_storage_igwo:
        continue
        
    real = results_storage_igwo[name]['real']
    pred = results_storage_igwo[name]['pred']
    
    for i, target_col in enumerate(target_vars):
        if i >= real.shape[1]: 
            break
        
        r_vals = real[:, i]
        p_vals = pred[:, i]
        
        mae = mean_absolute_error(r_vals, p_vals)
        mse = mean_squared_error(r_vals, p_vals)
        r2 = r2_score(r_vals, p_vals)
        
        mask = r_vals > 0.5 
        mape = np.mean(np.abs((r_vals[mask] - p_vals[mask]) / r_vals[mask])) * 100 if any(mask) else 0
        
        performance_report.append({
            'vessel': name,
            'engine': target_col.replace('ep_', '').replace('_FLW_SUP', ''),
            'mae (mt/h)': round(mae, 3),
            'mse (mt/h)^2': round(mse, 6),
            'r2 score': round(r2, 4),
            'mape (%)': round(mape, 2)
        })

if performance_report:
    print("\n" + "="*85)
    print(f"final test set performance audit for {TARGET_VESSEL.lower()} engine analytics using igwo")
    print("="*85)
    df_perf = pd.DataFrame(performance_report)
    print(df_perf.to_string(index=False))
    print("="*85)
    
    csv_filename = f"performance_testset_{TARGET_VESSEL.replace(' ', '_')}_igwo.csv"
    df_perf.to_csv(csv_filename, index=False)
    print(f"saved test set performance metrics to {csv_filename}")

for name in [TARGET_VESSEL]:
    if name not in results_storage_igwo: 
        continue
        
    real = results_storage_igwo[name]['real']
    pred = results_storage_igwo[name]['pred']
    
    n_engines = real.shape[1]
    fig, axes = plt.subplots(n_engines, 1, figsize=(15, 5 * n_engines), sharex=True)
    
    if n_engines == 1: 
        axes = [axes]
    
    for i in range(n_engines):
        engine_label = target_vars[i].replace('ep_', '').replace('_FLW_SUP', '')
        
        axes[i].plot(real[:, i], color='royalblue', label=f'actual {engine_label.lower()}', alpha=0.5)
        axes[i].plot(pred[:, i], color='crimson', label=f'lstm pred {engine_label.lower()}', linestyle='--', lw=1.5)
        
        axes[i].set_title(f"test set metrics for {name.lower()} with engine {engine_label.lower()} using igwo", fontweight='bold')
        axes[i].set_ylabel("mt/h")
        axes[i].legend(loc='upper right')
        axes[i].grid(True, linestyle='--', alpha=0.3)
    
    axes[-1].set_xlabel("time steps at fifteen minutes intervals")
    plt.tight_layout()
    
    plot_filename = f"test_plot_{name.replace(' ', '_')}_igwo.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"saved test set plot to {plot_filename}")
    
    plt.show()


### ECO Livorno

In [ ]:
TARGET_VESSEL = "ECO LIVORNO"

if 'results_storage_igwo' not in globals():
    results_storage_igwo = {}

for name in [TARGET_VESSEL]:
    if name not in lstm_data: 
        print(f"warning: {name} is not present in lstm_data skipping computation")
        continue
    
    print(f"\nbuilding and training igwo optimized model for {name.lower()}")
    
    d = lstm_data[name]
    x_tr, y_tr = d['x_train'], d['y_train']
    x_vl, y_vl = d['x_val'], d['y_val']
    x_ts, y_ts = d['x_test'], d['y_test']
    
    n_outputs = y_tr.shape[1] 
    
    param_file = f"opt_params_{name.replace(' ', '_')}.json"
    
    if os.path.exists(param_file):
        with open(param_file, "r") as f:
            p = json.load(f)
        u, lr, bs, dr = p["lstm_units"], p["learning_rate"], p["batch_size"], p["dropout_rate"]
        print(f"optimal parameters loaded from {param_file}: units={u}, lr={lr:.5f}, batch={bs}, dropout={dr:.2f}")
    else:
        print(f"optimization parameters not found at {param_file} using default configuration")
        u, lr, bs, dr = 64, 0.001, 32, 0.2

    model = Sequential([
        Input(shape=(x_tr.shape[1], x_tr.shape[2])),
        LSTM(u, return_sequences=True, activation='tanh'),
        Dropout(dr),
        LSTM(int(u/2), activation='tanh'),
        Dropout(dr),
        Dense(16, activation='relu'),
        Dense(n_outputs, activation='linear')
    ])

    model.compile(optimizer=Adam(learning_rate=lr), loss='huber', metrics=['mae'])

    print(f"starting igwo training for {name.lower()} on {len(x_tr)} sequences")
    stop = EarlyStopping(monitor='val_loss', patience=90, restore_best_weights=True)

    history = model.fit(
        x_tr, y_tr,
        validation_data=(x_vl, y_vl),
        epochs=100,
        batch_size=bs,
        callbacks=[stop],
        verbose=1
    )

    pred_scaled = model.predict(x_ts)
    scaler_y = vessel_scalers[name]['scaler_x'] if 'scaler_y' not in vessel_scalers[name] else vessel_scalers[name]['scaler_y']
    
    pred_real = scaler_y.inverse_transform(pred_scaled)
    real_real = scaler_y.inverse_transform(y_ts)

    results_storage_igwo[name] = {
        'real': real_real, 
        'pred': pred_real,
        'history': history.history
    }
    
    model_save_path = f"model_{name.replace(' ', '_')}_igwo.keras"
    model.save(model_save_path)
    print(f"model successfully saved to {model_save_path}")

performance_report = []
for name in [TARGET_VESSEL]:
    if name not in results_storage_igwo:
        continue
        
    real = results_storage_igwo[name]['real']
    pred = results_storage_igwo[name]['pred']
    
    for i, target_col in enumerate(target_vars):
        if i >= real.shape[1]: 
            break
        
        r_vals = real[:, i]
        p_vals = pred[:, i]
        
        mae = mean_absolute_error(r_vals, p_vals)
        mse = mean_squared_error(r_vals, p_vals)
        r2 = r2_score(r_vals, p_vals)
        
        mask = r_vals > 0.5 
        mape = np.mean(np.abs((r_vals[mask] - p_vals[mask]) / r_vals[mask])) * 100 if any(mask) else 0
        
        performance_report.append({
            'vessel': name,
            'engine': target_col.replace('ep_', '').replace('_FLW_SUP', ''),
            'mae (mt/h)': round(mae, 3),
            'mse (mt/h)^2': round(mse, 6),
            'r2 score': round(r2, 4),
            'mape (%)': round(mape, 2)
        })

if performance_report:
    print("\n" + "="*85)
    print(f"final performance audit for {TARGET_VESSEL.lower()} engine analytics using igwo")
    print("="*85)
    df_perf = pd.DataFrame(performance_report)
    print(df_perf.to_string(index=False))
    print("="*85)
    
    csv_filename = f"performance_{TARGET_VESSEL.replace(' ', '_')}_igwo.csv"
    df_perf.to_csv(csv_filename, index=False)
    print(f"saved performance metrics to {csv_filename}")

for name in [TARGET_VESSEL]:
    if name not in results_storage_igwo: 
        continue
        
    real = results_storage_igwo[name]['real']
    pred = results_storage_igwo[name]['pred']
    
    n_engines = real.shape[1]
    fig, axes = plt.subplots(n_engines, 1, figsize=(15, 5 * n_engines), sharex=True)
    
    if n_engines == 1: 
        axes = [axes]
    
    for i in range(n_engines):
        engine_label = target_vars[i].replace('ep_', '').replace('_FLW_SUP', '')
        
        axes[i].plot(real[:, i], color='royalblue', label=f'actual {engine_label.lower()}', alpha=0.5)
        axes[i].plot(pred[:, i], color='crimson', label=f'lstm pred {engine_label.lower()}', linestyle='--', lw=1.5)
        
        axes[i].set_title(f"final test set performance audit for{name.lower()} with engine {engine_label.lower()} using igwo", fontweight='bold')
        axes[i].set_ylabel("mt/h")
        axes[i].legend(loc='upper right')
        axes[i].grid(True, linestyle='--', alpha=0.3)
    
    axes[-1].set_xlabel("time steps at fifteen minutes intervals")
    plt.tight_layout()
    
    plot_filename = f"validation_plot_{name.replace(' ', '_')}_igwo.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"saved validation plot to {plot_filename}")
    
    plt.show()


## PREDICTION REPORT

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)

def get_mape(y_true, y_pred):
    mask = y_true > 0.05
    if np.sum(mask) == 0: return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

vessels_to_analyze = ["ECO ADRIATICA", "ECO LIVORNO"]

tab_buttons = ""
tab_contents = ""
first_tab = True

for vessel in vessels_to_analyze:
    if 'results_storage_igwo' not in globals() or vessel not in results_storage_igwo:
        continue
        
    real = results_storage_igwo[vessel]['real']
    pred = results_storage_igwo[vessel]['pred']
    
    # --- NOVITÀ: RECUPERO IL VOYAGE_ID INCROCIANDO I TIMESTAMP DEL TEST SET ---
    ts_array = lstm_data[vessel]['ts_test']
    df_ts = pd.DataFrame({'ts': pd.to_datetime(ts_array)})
    
    df_orig = datasets[vessel][['ts', 'Voyage_ID']].copy()
    df_orig['ts'] = pd.to_datetime(df_orig['ts'])
    
    # Merge per affiancare il Voyage_ID corretto a ogni punto temporale
    df_merged = pd.merge(df_ts, df_orig, on='ts', how='left')
    voyage_ids = df_merged['Voyage_ID'].fillna("Unknown").values
    # --------------------------------------------------------------------------
    
    n_engines = real.shape[1]
    
    performance_report = []
    for i in range(n_engines):
        engine_label = f"ME_{i+1}"
        y_r = real[:, i]
        y_p = pred[:, i]
        
        performance_report.append({
            'vessel': vessel,
            'engine': engine_label,
            'mae (mt/h)': round(mean_absolute_error(y_r, y_p), 3),
            'mse (mt/h)^2': round(mean_squared_error(y_r, y_p), 6),
            'r2 score': round(r2_score(y_r, y_p), 4),
            'mape (%)': round(get_mape(y_r, y_p), 2)
        })
        
    df_perf = pd.DataFrame(performance_report)
    stats_table_html = df_perf.to_html(index=False, justify='center', border=0, classes='stats-table')
    
    graphs_html = ""
    steps = list(range(1, len(real) + 1))
    
    step_skip = 1 if len(steps) < 3000 else (len(steps)//1500)
    
    for i in range(n_engines):
        engine_label = f"me_{i+1}"
        fig = go.Figure()
        
        # Testo dinamico per l'hover che contiene il Voyage ID
        hover_text_real = [f"Voyage: {v}" for v in voyage_ids[::step_skip]]
        
        fig.add_trace(go.Scatter(
            x=steps[::step_skip], 
            y=real[::step_skip, i], 
            mode='lines', 
            name=f'actual {engine_label}', 
            line=dict(color='royalblue', width=1.5),
            text=hover_text_real,
            hovertemplate='<b>%{text}</b><br>Step: %{x}<br>Actual: %{y:.3f} mt/h<extra></extra>'
        ))
        
        fig.add_trace(go.Scatter(
            x=steps[::step_skip], 
            y=pred[::step_skip, i], 
            mode='lines', 
            name=f'lstm pred {engine_label}', 
            line=dict(color='crimson', width=2, dash='dash'),
            text=hover_text_real,
            hovertemplate='<b>%{text}</b><br>Step: %{x}<br>Pred: %{y:.3f} mt/h<extra></extra>'
        ))
        
        fig.update_layout(
            title=f"test set validation for {vessel.lower()} with engine {engine_label} using igwo",
            xaxis_title="time steps at fifteen minutes intervals",
            yaxis_title="mt/h",
            template="plotly_dark", 
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
            font=dict(color='#c9d1d9', family="monospace"), 
            margin=dict(l=40, r=40, t=50, b=40),
            hovermode="x unified"
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(255,255,255,0.1)', griddash='dash')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(255,255,255,0.1)', griddash='dash')
        
        graphs_html += f'<div style="margin-bottom: 30px; border: 1px solid #30363d; border-radius: 8px; width: 100%;">{fig.to_html(full_html=False, include_plotlyjs=False)}</div>'

    safe_id = vessel.replace(" ", "_")
    active_class = "active" if first_tab else ""
    tab_buttons += f"<button class='tablinks {active_class}' onclick=\"openVessel(event, '{safe_id}')\">{vessel}</button>\n"
    
    display_style = "block" if first_tab else "none"
    tab_contents += f"""
    <div id='{safe_id}' class='tabcontent' style='display: {display_style};'>
        <h3>final performance audit for {vessel.lower()} engine analytics using igwo</h3>
        {stats_table_html}
        <br>
        {graphs_html}
    </div>
    """
    first_tab = False

final_html = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <title>LSTM IGWO Performance Audit</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: 'Courier New', Courier, monospace; background-color: #0d1117; color: #c9d1d9; padding: 40px; margin: 0; }}
        .container {{ max-width: 1400px; margin: auto; background-color: #161b22; padding: 40px; border-radius: 12px; border: 1px solid #30363d; box-shadow: 0 10px 20px rgba(0,0,0,0.6); }}
        h1 {{ color: #58a6ff; text-align: center; font-weight: 800; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 30px; font-family: 'Segoe UI', 'Inter', sans-serif; }}
        h3 {{ color: #c9d1d9; margin-top: 40px; font-weight: bold; border-bottom: 1px dashed #30363d; padding-bottom: 10px; text-transform: lowercase; }}
        
        .stats-table {{ width: 100%; border-collapse: collapse; margin: 20px 0; background-color: #0d1117; border-radius: 6px; overflow: hidden; }}
        .stats-table th, .stats-table td {{ padding: 12px 15px; text-align: center; font-size: 1.1em; }}
        .stats-table th {{ color: #8b949e; font-weight: bold; border-bottom: 1px dashed #30363d; border-top: 1px dashed #30363d; }}
        .stats-table td {{ border-bottom: 1px dashed #30363d; }}
        
        .tab {{ overflow: hidden; border-bottom: 2px solid #30363d; margin-bottom: 20px; display: flex; justify-content: center; font-family: 'Segoe UI', 'Inter', sans-serif; }}
        .tab button {{ background-color: #0d1117; color: #8b949e; float: left; border: none; outline: none; cursor: pointer; padding: 14px 30px; transition: 0.3s; font-size: 1.1em; font-weight: bold; border-top-left-radius: 8px; border-top-right-radius: 8px; margin: 0 5px; }}
        .tab button:hover {{ background-color: #1c2128; color: #c9d1d9; }}
        .tab button.active {{ background-color: #1f2937; color: #58a6ff; border-bottom: 3px solid #58a6ff; }}
        .tabcontent {{ padding: 20px 0; animation: fadeEffect 0.5s; width: 100%; }}
        @keyframes fadeEffect {{ from {{opacity: 0;}} to {{opacity: 1;}} }}
        
        .plotly-graph-div {{ width: 100% !important; }}
    </style>
    <script>
        function openVessel(evt, vesselName) {{
            var i, tabcontent, tablinks;
            tabcontent = document.getElementsByClassName("tabcontent");
            for (i = 0; i < tabcontent.length; i++) {{ tabcontent[i].style.display = "none"; }}
            tablinks = document.getElementsByClassName("tablinks");
            for (i = 0; i < tablinks.length; i++) {{ tablinks[i].className = tablinks[i].className.replace(" active", ""); }}
            document.getElementById(vesselName).style.display = "block";
            evt.currentTarget.className += " active";
            
            // LA MAGIA E' QUI: Questo forza Plotly a ricalcolare la larghezza 100% quando la tab diventa visibile!
            window.dispatchEvent(new Event('resize')); 
        }}
    </script>
</head>
<body>
    <div class='container'>
        <h1>LSTM IGWO Performance Audit</h1>
        <div class="tab">
            {tab_buttons}
        </div>
        {tab_contents}
    </div>
</body>
</html>
"""

report_path = r"C:\Users\siani\LSTM\LSTM_Performance_Report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"\nReport saved to: {report_path}")


## SCALA BEAUFORT DEFINITION

In [ ]:
# Extract the dataset for ECO ADRIATICA specifically
df_historical = datasets["ECO ADRIATICA"].copy()

# Note: The trigonometric transformation (SIN/COS) is skipped here 
# because it was already applied during data loading.

weather_features_to_save = [
    'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1', 'ep_WH_SWELLP_1', 
    'ep_WH_SPEED_1', 'ep_SHIP_SEAF_1', 'ep_WH_AIRT_1',
    'ep_WH_DIR_1_SIN', 'ep_WH_DIR_1_COS', 
    'ep_WH_WAVED_1_SIN', 'ep_WH_WAVED_1_COS',
    'ep_WH_SWELLD_1_SIN', 'ep_WH_SWELLD_1_COS',
    'ep_SHIP_SEADIR_1_SIN', 'ep_SHIP_SEADIR_1_COS'
]

available_features = [f for f in weather_features_to_save if f in df_historical.columns]

# Drop NaNs to ensure clean calculations
df_clean = df_historical.dropna(subset=['SHA_POW_TOT'] + available_features).copy()

# Split the dataset into 13 quantiles based purely on real power consumption
df_clean['Consumption_Level'] = pd.qcut(df_clean['SHA_POW_TOT'], q=13, labels=False)

global_beaufort_consumption_scale = {}

for level in range(13):
    df_level = df_clean[df_clean['Consumption_Level'] == level]
    
    centroid = {}
    for f in available_features:
        centroid[f] = df_level[f].mean()
        
    # Save the mean shaft power for reference in the visualization
    centroid['SHA_POW_TOT'] = df_level['SHA_POW_TOT'].mean()
    
    global_beaufort_consumption_scale[float(level)] = centroid

scale_path = r"C:\Users\siani\LSTM\global_beaufort_consumption_scale.pkl"
joblib.dump(global_beaufort_consumption_scale, scale_path)
print(f"13-Level scale successfully saved in: {scale_path}")

df_scale = pd.DataFrame.from_dict(global_beaufort_consumption_scale, orient='index')
df_scale.index.name = 'Beaufort-Style Level'
df_scale.reset_index(inplace=True)

# Display only the most readable columns to prevent clutter
cols_to_show = ['Beaufort-Style Level', 'SHA_POW_TOT', 'ep_WH_WAVEH_1', 'ep_WH_SPEED_1', 'ep_SHIP_SEAF_1']
cols_present = [c for c in cols_to_show if c in df_scale.columns]

styled_scale = df_scale[cols_present].style.background_gradient(
    subset=['SHA_POW_TOT'], cmap='YlOrRd'
).format(precision=2)

display(styled_scale)


In [ ]:
print("Assigning Beaufort consumption level to each historical record...")

df_clean['Consumption_Level'] = pd.qcut(df_clean['SHA_POW_TOT'], q=13, labels=False)

df_historical_labeled = df_clean.copy()
df_historical_labeled['Beaufort_Level'] = df_historical_labeled['Consumption_Level'].astype(int)

output_path = r"C:\Users\siani\LSTM\df_historical_beaufort_labeled.pkl"
joblib.dump(df_historical_labeled, output_path)
print(f"Labeled historical dataset saved in: {output_path}")

print("\n=== BEAUFORT LEVEL DISTRIBUTION IN HISTORICAL DATA ===")
distribution = df_historical_labeled['Beaufort_Level'].value_counts().sort_index()
for level, count in distribution.items():
    pct = count / len(df_historical_labeled) * 100
    print(f"  Force {level:2d}: {count:6,} records  ({pct:.1f}%)")

print(f"\nTotal labeled records: {len(df_historical_labeled):,}")
print("Ready for inverse lookup during scheduling.")


## SCHEDULE ON 23 FEB - 02 MAR 


### INITIALIZATION, SCHEDULE DATA, AND WEATHER CLUSTERING

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import warnings
import tensorflow as tf
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

schedule_data = [
    {
        "departure_port": "Valencia",  "arrival_port": "Barcelona",
        "departure_dt": "2026-02-23 19:00", "arrival_dt": "2026-02-24 06:42",
        "distance_nm": 158.46, "time_at_sea_h": 11.7,
        "draft_aft": 6.6,  "draft_fore": 6.3,  "avg_speed": 16.01,
        "real_fuel_mt": 18.2,  "beaufort_level": 1
    },
    {
        "departure_port": "Barcelona", "arrival_port": "Livorno",
        "departure_dt": "2026-02-24 10:30", "arrival_dt": "2026-02-25 07:36",
        "distance_nm": 388.23, "time_at_sea_h": 21.1,
        "draft_aft": 7.0,  "draft_fore": 6.7,  "avg_speed": 20.45,
        "real_fuel_mt": 64.5,  "beaufort_level": 1
    },
    {
        "departure_port": "Livorno",   "arrival_port": "Savona",
        "departure_dt": "2026-02-25 16:30", "arrival_dt": "2026-02-25 22:54",
        "distance_nm": 89.36,  "time_at_sea_h": 6.4,
        "draft_aft": 6.7,  "draft_fore": 6.4,  "avg_speed": 20.38,
        "real_fuel_mt": 14.6,  "beaufort_level": 1
    },
    {
        "departure_port": "Savona",    "arrival_port": "Barcelona",
        "departure_dt": "2026-02-26 05:42", "arrival_dt": "2026-02-27 01:00",
        "distance_nm": 344.53, "time_at_sea_h": 19.3,
        "draft_aft": 6.7,  "draft_fore": 6.5,  "avg_speed": 20.28,
        "real_fuel_mt": 51.7,  "beaufort_level": 1
    },
    {
        "departure_port": "Barcelona", "arrival_port": "Valencia",
        "departure_dt": "2026-02-27 04:54", "arrival_dt": "2026-02-27 17:42",
        "distance_nm": 157.37, "time_at_sea_h": 12.8,
        "draft_aft": 6.5,  "draft_fore": 6.3,  "avg_speed": 14.74,
        "real_fuel_mt": 17.6,  "beaufort_level": 1
    },
    {
        "departure_port": "Valencia",  "arrival_port": "Savona",
        "departure_dt": "2026-02-28 12:48", "arrival_dt": "2026-03-01 22:18",
        "distance_nm": 506.65, "time_at_sea_h": 33.5,
        "draft_aft": 6.9,  "draft_fore": 6.6,  "avg_speed": 15.74,
        "real_fuel_mt": 61.2,  "beaufort_level": 1
    },
    {
        "departure_port": "Savona",    "arrival_port": "Livorno",
        "departure_dt": "2026-03-02 12:18", "arrival_dt": "2026-03-02 20:18",
        "distance_nm": 84.35,  "time_at_sea_h": 8.0,
        "draft_aft": 6.1,  "draft_fore": 6.0,  "avg_speed": 14.85,
        "real_fuel_mt": 10.9,  "beaufort_level": 1
    }
]


lstm_expected_inputs = [
    'ep_SHIP_SOG_1', 'ep_SHIP_STW_1', 'ep_SHIP_DRAFTAFT_1', 'ep_SHIP_DRAFTFOR_1',
    'ep_SHIP_HEAD_1', 'miglia', 'SHA_POW_TOT', 'SHG_POW_TOT',
    'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1', 'ep_WH_SWELLP_1',
    'ep_WH_SPEED_1', 'ep_WH_DIR_1', 'ep_WH_AIRT_1',
    'ep_SHIP_LAT_1', 'ep_SHIP_LON_1'
]

def beaufort_badge(level):
    if level <= 2:   bg, fg, label = "#d1fae5", "#065f46", "Calm"
    elif level <= 5: bg, fg, label = "#fef3c7", "#92400e", "Moderate"
    elif level <= 8: bg, fg, label = "#fed7aa", "#9a3412", "Strong"
    else:            bg, fg, label = "#fecaca", "#7f1d1d", "Storm"
    return f'<span style="background:{bg};color:{fg};font-weight:bold;padding:3px 8px;border-radius:12px;font-size:12px;">Force {level} – {label}</span>'

rows_html = ""
for i, leg in enumerate(schedule_data):
    row_bg = "#f3f4f6" if i % 2 == 0 else "#ffffff"
    rows_html += f"""
    <tr style="background:{row_bg};height:45px;border-bottom:1px solid #e5e7eb;">
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['departure_port']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['arrival_port']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['departure_dt']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['arrival_dt']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['distance_nm']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['time_at_sea_h']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['draft_aft']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['draft_fore']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;">{leg['avg_speed']}</td>
        <td style="padding:12px;border:1px solid #d1d5db;text-align:center;">{beaufort_badge(leg['beaufort_level'])}</td>
        <td style="padding:12px;border:1px solid #d1d5db;background:#fee2e2;font-weight:bold;color:#b91c1c;">{leg['real_fuel_mt']}</td>
    </tr>"""

html_table = f"""
<div style="overflow-x:auto;font-family:'Segoe UI',sans-serif;padding:20px;background:#fff;">
<table style="width:100%;border-collapse:collapse;min-width:1100px;font-size:14px;border:1px solid #d1d5db;">
  <caption><strong style="font-size:16px;color:#374151;display:block;margin-bottom:10px;">
    Voyage Schedule – February 2026 – ECO ADRIATICA</strong></caption>
  <thead>
    <tr style="background:#d1d5db;border-bottom:2px solid #9ca3af;height:50px;">
      <th style="padding:12px;border:1px solid #9ca3af;">DEPARTURE PORT</th>
      <th style="padding:12px;border:1px solid #9ca3af;">ARRIVAL PORT</th>
      <th style="padding:12px;border:1px solid #9ca3af;width:140px;">DEPARTURE</th>
      <th style="padding:12px;border:1px solid #9ca3af;width:140px;">ARRIVAL</th>
      <th style="padding:12px;border:1px solid #9ca3af;">DIST (nm)</th>
      <th style="padding:12px;border:1px solid #9ca3af;">TIME (h)</th>
      <th style="padding:12px;border:1px solid #9ca3af;">DRAFT AFT</th>
      <th style="padding:12px;border:1px solid #9ca3af;">DRAFT FOR</th>
      <th style="padding:12px;border:1px solid #9ca3af;">AVG SPD (kn)</th>
      <th style="padding:12px;border:1px solid #9ca3af;background:#dbeafe;text-align:center;">WEATHER</th>
      <th style="padding:12px;border:1px solid #9ca3af;background:#fca5a5;">HFO ME (MT)</th>
    </tr>
  </thead>
  <tbody style="color:#4b5563;">{rows_html}</tbody>
</table></div>"""

with open(r"C:\Users\siani\Desktop\table_schedule.html", "w", encoding="utf-8") as f:
    f.write(html_table)

display(HTML(html_table))
print(f"Schedule loaded: {len(schedule_data)} legs ready.")


DEPARTURE PORT,ARRIVAL PORT,DEPARTURE,ARRIVAL,DIST (nm),TIME (h),DRAFT AFT,DRAFT FOR,AVG SPD (kn),WEATHER,HFO ME (MT)
Valencia,Barcelona,2026-02-23 19:00,2026-02-24 06:42,158.46,11.7,6.6,6.3,16.01,Force 1 – Calm,18.2
Barcelona,Livorno,2026-02-24 10:30,2026-02-25 07:36,388.23,21.1,7.0,6.7,20.45,Force 1 – Calm,64.5
Livorno,Savona,2026-02-25 16:30,2026-02-25 22:54,89.36,6.4,6.7,6.4,20.38,Force 1 – Calm,14.6
Savona,Barcelona,2026-02-26 05:42,2026-02-27 01:00,344.53,19.3,6.7,6.5,20.28,Force 1 – Calm,51.7
Barcelona,Valencia,2026-02-27 04:54,2026-02-27 17:42,157.37,12.8,6.5,6.3,14.74,Force 1 – Calm,17.6
Valencia,Savona,2026-02-28 12:48,2026-03-01 22:18,506.65,33.5,6.9,6.6,15.74,Force 1 – Calm,61.2
Savona,Livorno,2026-03-02 12:18,2026-03-02 20:18,84.35,8.0,6.1,6.0,14.85,Force 1 – Calm,10.9


Schedule loaded: 7 legs ready.


### Physical Rules Extraction Inverse Lookup

In [2]:
print("Extracting physical rules from historical data via inverse Beaufort lookup...")

scale_path   = r"C:\Users\siani\LSTM\global_beaufort_consumption_scale.pkl"
labeled_path = r"C:\Users\siani\LSTM\df_historical_beaufort_labeled.pkl"

global_beaufort_scale = joblib.load(scale_path)
df_hist = joblib.load(labeled_path)

route_physics = {}

for leg in schedule_data:
    dep = leg['departure_port'].upper()
    arr = leg['arrival_port'].upper()
    route_str = f"{dep} to {arr}"
    route_key = f"{dep}_{arr}"
    beaufort_target = leg['beaufort_level']
    avg_speed = leg['avg_speed']

    df_route = df_hist[df_hist['Route'] == route_str].copy()

    if df_route.empty:
        print(f"  ⚠️  No historical data for route {route_str}. Skipping.")
        continue

    # Step 1: Hydrodynamic constant c from calm sea records (Beaufort 0 or 1)
    df_calm = df_route[df_route['Beaufort_Level'].isin([0, 1])].copy()
    df_calm = df_calm[
        (df_calm['ep_SHIP_SOG_1'] > 5) &
        (df_calm['SHA_POW_TOT'] > 100)
    ].copy()

    if len(df_calm) >= 10:
        sog3 = df_calm['ep_SHIP_SOG_1'] ** 3
        c_values = df_calm['SHA_POW_TOT'] / sog3.replace(0, np.nan)
        c = np.nanmedian(c_values)
    else:
        df_calm_all = df_hist[df_hist['Beaufort_Level'].isin([0, 1])].copy()
        df_calm_all = df_calm_all[df_calm_all['ep_SHIP_SOG_1'] > 5]
        c = np.nanmedian(df_calm_all['SHA_POW_TOT'] / (df_calm_all['ep_SHIP_SOG_1'] ** 3))
        print(f"  ℹ️  Using global c for {route_str}")

    # Step 2: Weather penalty factor for the target Beaufort level
    df_target_level = df_route[df_route['Beaufort_Level'] == beaufort_target].copy()
    df_target_level = df_target_level[df_target_level['ep_SHIP_SOG_1'] > 5]

    if len(df_target_level) >= 5:
        power_at_target = np.nanmedian(df_target_level['SHA_POW_TOT'])
        power_at_calm   = c * (avg_speed ** 3)
        penalty_factor  = power_at_target / power_at_calm if power_at_calm > 0 else 1.0
        penalty_factor  = np.clip(penalty_factor, 1.0, 3.5)
    else:
        calm_centroid   = global_beaufort_scale.get(1.0, global_beaufort_scale[min(global_beaufort_scale)])
        target_centroid = global_beaufort_scale.get(float(beaufort_target), calm_centroid)
        penalty_factor  = target_centroid.get('SHA_POW_TOT', 1.0) / max(calm_centroid.get('SHA_POW_TOT', 1.0), 1)
        penalty_factor  = np.clip(penalty_factor, 1.0, 3.5)
        print(f"  ℹ️  Using centroid penalty for {route_str} at Beaufort {beaufort_target}")

    # Step 3: Noise level
    if len(df_target_level) >= 5:
        noise_std = df_target_level['SHA_POW_TOT'].std() / max(df_target_level['SHA_POW_TOT'].mean(), 1)
        noise_std = np.clip(noise_std, 0.01, 0.25)
    else:
        noise_std = 0.05

    # Step 4: SHG electrical baseline (no ts column available, use all route records)
    if 'SHG_POW_TOT' in df_route.columns and df_route['SHG_POW_TOT'].notna().any():
        shg_baseline = np.nanmedian(df_route['SHG_POW_TOT'])
    elif 'SHG_POW_TOT' in df_hist.columns:
        shg_baseline = np.nanmedian(df_hist['SHG_POW_TOT'])
    else:
        shg_baseline = 0.0

    # Step 5: Slippage (SOG - STW) from calm records
    if 'ep_SHIP_STW_1' in df_calm.columns and len(df_calm) >= 10:
        slip_mean = np.nanmedian(df_calm['ep_SHIP_SOG_1'] - df_calm['ep_SHIP_STW_1'])
        slip_mean = np.clip(slip_mean, 0, 2.0)
    else:
        slip_mean = 0.3

    # Step 6: Warm-up block (16 consecutive records at target Beaufort on this route)
    df_warmup_pool = df_route[df_route['Beaufort_Level'] == beaufort_target].copy()
    if len(df_warmup_pool) < 16:
        available = df_route['Beaufort_Level'].dropna().unique()
        nearest_b = min(available, key=lambda x: abs(x - beaufort_target)) if len(available) > 0 else 0
        df_warmup_pool = df_route[df_route['Beaufort_Level'] == nearest_b].copy()
        print(f"  ℹ️  Warm-up using nearest Beaufort {nearest_b} for {route_str}")

    df_warmup = pd.DataFrame()
    if 'Voyage_ID' in df_warmup_pool.columns:
        best_v, min_err = None, float('inf')
        for v in df_warmup_pool['Voyage_ID'].unique():
            df_v = df_warmup_pool[df_warmup_pool['Voyage_ID'] == v]
            if len(df_v) < 16: continue
            err = abs(df_v['ep_SHIP_SOG_1'].mean() - avg_speed)
            if err < min_err:
                min_err = err
                best_v = v
        if best_v:
            df_warmup = df_warmup_pool[df_warmup_pool['Voyage_ID'] == best_v].tail(16).copy()

    if len(df_warmup) < 16:
        df_warmup = df_warmup_pool.tail(16).copy()

    # Weather centroid for this Beaufort level
    bft_key = float(beaufort_target)
    if bft_key not in global_beaufort_scale:
        bft_key = min(global_beaufort_scale.keys(), key=lambda k: abs(k - beaufort_target))
    weather_centroid = global_beaufort_scale[bft_key]

    route_physics[route_key] = {
        "c":                c,
        "penalty_factor":   penalty_factor,
        "noise_std":        noise_std,
        "shg_baseline":     shg_baseline,
        "slip_mean":        slip_mean,
        "df_warmup":        df_warmup,
        "weather_centroid": weather_centroid
    }

    print(f"✅ {route_str:35s} | Beaufort {beaufort_target:2d} | c={c:.2f} | penalty={penalty_factor:.3f} | noise={noise_std:.3f}")

print(f"\nPhysical rules extracted for {len(route_physics)} routes.")


Extracting physical rules from historical data via inverse Beaufort lookup...
✅ VALENCIA to BARCELONA               | Beaufort  1 | c=1.90 | penalty=1.006 | noise=0.013
  ℹ️  Using global c for BARCELONA to LIVORNO
  ℹ️  Using centroid penalty for BARCELONA to LIVORNO at Beaufort 1
  ℹ️  Warm-up using nearest Beaufort 0 for BARCELONA to LIVORNO
✅ BARCELONA to LIVORNO                | Beaufort  1 | c=1.91 | penalty=1.000 | noise=0.050
✅ LIVORNO to SAVONA                   | Beaufort  1 | c=1.73 | penalty=1.000 | noise=0.016
✅ SAVONA to BARCELONA                 | Beaufort  1 | c=1.81 | penalty=1.000 | noise=0.012
✅ BARCELONA to VALENCIA               | Beaufort  1 | c=1.89 | penalty=1.301 | noise=0.012
✅ VALENCIA to SAVONA                  | Beaufort  1 | c=2.12 | penalty=1.000 | noise=0.014
✅ SAVONA to LIVORNO                   | Beaufort  1 | c=1.91 | penalty=1.253 | noise=0.013

Physical rules extracted for 7 routes.


### Timeline Initialization

In [3]:
print("Initializing simulation timelines and pre-allocating vectors...\n")

sim_registry = {}

for leg in schedule_data:
    dep = leg['departure_port'].upper()
    arr = leg['arrival_port'].upper()
    route_key = f"{dep}_{arr}"
    route_str = f"{dep} to {arr}"

    if route_key not in route_physics:
        print(f"  ⚠️  No physics for {route_str}. Skipping.")
        continue

    n_rows = int(leg['time_at_sea_h'] * 4)
    time_index = pd.date_range(
        start=leg['departure_dt'],
        periods=n_rows,
        freq='15min'
    )

    df_sim = pd.DataFrame(index=range(n_rows))
    df_sim['ts'] = time_index

    # Pre-populate constant columns (same value for all rows)
    df_sim['ep_SHIP_DRAFTAFT_1'] = leg['draft_aft']
    df_sim['ep_SHIP_DRAFTFOR_1'] = leg['draft_fore']

    # Allocate empty columns to be filled by the generation loop
    for col in ['ep_SHIP_SOG_1', 'ep_SHIP_STW_1', 'miglia',
                'SHA_POW_TOT', 'SHG_POW_TOT', 'ep_SHIP_HEAD_1',
                'ep_SHIP_LAT_1', 'ep_SHIP_LON_1']:
        df_sim[col] = np.nan

    # Interpolate positional columns from historical mean (best available)
    phys = route_physics[route_key]
    df_wu = phys['df_warmup']

    for col in ['ep_SHIP_HEAD_1', 'ep_SHIP_LAT_1', 'ep_SHIP_LON_1']:
        if col in df_wu.columns and df_wu[col].notna().any():
            mean_val = df_wu[col].mean()
            df_sim[col] = mean_val
        else:
            df_sim[col] = 0.0

    sim_registry[route_key] = {
        "leg":    leg,
        "n_rows": n_rows,
        "df_sim": df_sim
    }

    print(f"✅ {route_str:35s} | {n_rows} rows @ 15-min steps ({leg['time_at_sea_h']}h)")

print(f"\nTimelines initialized for {len(sim_registry)} routes.")


Initializing simulation timelines and pre-allocating vectors...

✅ VALENCIA to BARCELONA               | 46 rows @ 15-min steps (11.7h)
✅ BARCELONA to LIVORNO                | 84 rows @ 15-min steps (21.1h)
✅ LIVORNO to SAVONA                   | 25 rows @ 15-min steps (6.4h)
✅ SAVONA to BARCELONA                 | 77 rows @ 15-min steps (19.3h)
✅ BARCELONA to VALENCIA               | 51 rows @ 15-min steps (12.8h)
✅ VALENCIA to SAVONA                  | 134 rows @ 15-min steps (33.5h)
✅ SAVONA to LIVORNO                   | 32 rows @ 15-min steps (8.0h)

Timelines initialized for 7 routes.


### Coherent Generation Loop

In [ ]:
print("Running coherent generation loop (dynamic weather profiles + route fallback)...\n")

np.random.seed(42)

ROUTE_FALLBACK = {
    "BARCELONA to LIVORNO": "BARCELONA to SAVONA",
    "LIVORNO to BARCELONA": "SAVONA to BARCELONA"
}

for route_key, entry in sim_registry.items():
    leg    = entry['leg']
    n_rows = entry['n_rows']
    df_sim = entry['df_sim']
    phys   = route_physics[route_key]

    avg_speed  = leg['avg_speed']
    slip_mean  = phys['slip_mean']
    shg_baseline = phys['shg_baseline']
    beaufort_target = leg['beaufort_level']

    dep = leg['departure_port'].upper()
    arr = leg['arrival_port'].upper()
    route_str = f"{dep} to {arr}"

    # Route loader with fallback
    df_route = df_hist[df_hist['Route'] == route_str].copy()
    if df_route.empty:
        fallback_route = ROUTE_FALLBACK.get(route_str, None)
        if fallback_route:
            df_route = df_hist[df_hist['Route'] == fallback_route].copy()
            print(f"  ℹ️  Route {route_str} not found in df_hist. Using fallback: {fallback_route}")
        else:
            print(f"  ⚠️  No historical data and no fallback defined for {route_str}. Skipping.")
            continue

    xp_new = np.linspace(0, 1, n_rows)

    # Step A: Find the kinematic voyage (SOG closest to schedule avg_speed)
    best_kin_v, min_err = None, float('inf')
    for v in df_route['Voyage_ID'].unique():
        df_v = df_route[df_route['Voyage_ID'] == v]
        if len(df_v) < 10: continue
        err = abs(df_v['ep_SHIP_SOG_1'].mean() - avg_speed)
        if err < min_err:
            min_err = err
            best_kin_v = v

    if not best_kin_v:
        print(f"  ⚠️  No valid kinematic voyage found for {route_str}. Skipping.")
        continue

    df_kin = df_route[df_route['Voyage_ID'] == best_kin_v].copy()
    xp_kin = np.linspace(0, 1, len(df_kin))

    # Step B: Find the weather voyage at the target Beaufort level
    df_bft = df_route[df_route['Beaufort_Level'] == beaufort_target].copy()

    if len(df_bft) < 10:
        available = df_route['Beaufort_Level'].dropna().unique()
        nearest_b = min(available, key=lambda x: abs(x - beaufort_target)) if len(available) > 0 else 0
        df_bft = df_route[df_route['Beaufort_Level'] == nearest_b].copy()
        print(f"  ℹ️  Using nearest Beaufort {nearest_b} for weather on {route_str}")

    best_wth_v, min_dist = None, float('inf')
    for v in df_bft['Voyage_ID'].unique():
        df_v = df_bft[df_bft['Voyage_ID'] == v]
        if len(df_v) < 10: continue
        dist = abs(df_v['ep_SHIP_SOG_1'].mean() - avg_speed)
        if dist < min_dist:
            min_dist = dist
            best_wth_v = v

    if not best_wth_v:
        best_wth_v = best_kin_v
        df_wth = df_kin.copy()
    else:
        df_wth = df_bft[df_bft['Voyage_ID'] == best_wth_v].copy()
        
    xp_wth = np.linspace(0, 1, len(df_wth))

    # Step C: Interpolate SOG, STW, and Slip
    real_sog = df_kin['ep_SHIP_SOG_1'].interpolate().ffill().bfill().values
    sim_sog  = np.interp(xp_new, xp_kin, real_sog)
    sim_sog  = sim_sog - np.mean(sim_sog) + avg_speed

    real_stw = df_kin['ep_SHIP_STW_1'].interpolate().ffill().bfill().values
    real_slip = real_sog - real_stw
    sim_slip  = np.interp(xp_new, xp_kin, real_slip)

    df_sim['ep_SHIP_SOG_1'] = sim_sog
    df_sim['ep_SHIP_STW_1'] = np.clip(sim_sog - sim_slip, 0.5, None)
    df_sim['miglia']        = sim_sog / 4.0

    for col in ['ep_SHIP_HEAD_1', 'ep_SHIP_LAT_1', 'ep_SHIP_LON_1']:
        if col in df_kin.columns:
            vals = df_kin[col].interpolate().ffill().bfill().values
            df_sim[col] = np.interp(xp_new, xp_kin, vals)

    # Step D: Interpolate Weather Dynamically from df_wth (NOT centroids)
    # This preserves the dynamic variability of wind and waves
    weather_cols = ['ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1',
                    'ep_WH_SWELLP_1', 'ep_WH_SPEED_1', 'ep_WH_AIRT_1']

    for w_col in weather_cols:
        if w_col in df_wth.columns:
            vals = df_wth[w_col].interpolate().ffill().bfill().values
            df_sim[w_col] = np.interp(xp_new, xp_wth, vals)

    # Dynamically reconstruction 
    if 'ep_WH_DIR_1_SIN' in df_wth.columns and 'ep_WH_DIR_1_COS' in df_wth.columns:
        sin_vals = df_wth['ep_WH_DIR_1_SIN'].interpolate().ffill().bfill().values
        cos_vals = df_wth['ep_WH_DIR_1_COS'].interpolate().ffill().bfill().values
        sim_sin = np.interp(xp_new, xp_wth, sin_vals)
        sim_cos = np.interp(xp_new, xp_wth, cos_vals)
        df_sim['ep_WH_DIR_1'] = (np.rad2deg(np.arctan2(sim_sin, sim_cos)) + 360) % 360
    elif 'ep_WH_DIR_1' in df_wth.columns:
        vals = df_wth['ep_WH_DIR_1'].interpolate().ffill().bfill().values
        df_sim['ep_WH_DIR_1'] = np.interp(xp_new, xp_wth, vals)
    else:
        df_sim['ep_WH_DIR_1'] = 0.0

    # Step E: SHA from weather voyage with speed correction
    if 'SHA_POW_TOT' in df_wth.columns:
        real_sha    = df_wth['SHA_POW_TOT'].interpolate().ffill().bfill().values
        sim_sha     = np.interp(xp_new, xp_wth, real_sha)
        real_sog_wth = df_wth['ep_SHIP_SOG_1'].interpolate().ffill().bfill().values
        sim_sog_wth = np.interp(xp_new, xp_wth, real_sog_wth)
        sim_sog_wth = np.where(sim_sog_wth < 5.0, 5.0, sim_sog_wth)

        # Dynamic cubic correction
        speed_ratio = df_sim['ep_SHIP_SOG_1'] / sim_sog_wth
        speed_ratio = np.clip(speed_ratio, 0.7, 1.5)
        df_sim['SHA_POW_TOT'] = sim_sha * (speed_ratio ** 3)
    else:
        df_sim['SHA_POW_TOT'] = np.full(n_rows, phys['c'] * (avg_speed ** 3))

    if 'SHG_POW_TOT' in df_wth.columns:
        real_shg = df_wth['SHG_POW_TOT'].interpolate().ffill().bfill().values
        df_sim['SHG_POW_TOT'] = np.interp(xp_new, xp_wth, real_shg)
    else:
        df_sim['SHG_POW_TOT'] = shg_baseline

    speed_err = abs(df_sim['ep_SHIP_SOG_1'].mean() - avg_speed)
    assert speed_err < 0.05, f"Speed mean error: {speed_err:.4f}"

    entry['df_sim'] = df_sim
    print(f"✅ {route_key:30s} | mean SOG={df_sim['ep_SHIP_SOG_1'].mean():.3f} kn | "
          f"SHA mean={df_sim['SHA_POW_TOT'].mean():.0f} kW (fully dynamic weather)")

print("\nGeneration loop complete. Weather data is now fully dynamic.")


Running coherent generation loop (dynamic weather profiles + route fallback)...

✅ VALENCIA_BARCELONA             | mean SOG=16.010 kn | SHA mean=7789 kW (fully dynamic weather)
  ℹ️  Using nearest Beaufort 0 for weather on BARCELONA to LIVORNO
✅ BARCELONA_LIVORNO              | mean SOG=20.450 kn | SHA mean=17186 kW (fully dynamic weather)
✅ LIVORNO_SAVONA                 | mean SOG=20.380 kn | SHA mean=14397 kW (fully dynamic weather)
✅ SAVONA_BARCELONA               | mean SOG=20.280 kn | SHA mean=12633 kW (fully dynamic weather)
✅ BARCELONA_VALENCIA             | mean SOG=14.740 kn | SHA mean=6465 kW (fully dynamic weather)
✅ VALENCIA_SAVONA                | mean SOG=15.740 kn | SHA mean=7633 kW (fully dynamic weather)
✅ SAVONA_LIVORNO                 | mean SOG=14.850 kn | SHA mean=7327 kW (fully dynamic weather)

Generation loop complete. Weather data is now fully dynamic.


### Concatenation and LSTM Windowing

In [ ]:
# CELLA 5: ENSEMBLE GENERATION

warnings.filterwarnings('ignore')
print("Generazione dei profili temporali dinamici (Integrità Geo-Fisica pura, Nessun Fallback)...")

lstm_expected_inputs = [
    'ep_SHIP_SOG_1', 'ep_SHIP_STW_1', 'ep_SHIP_DRAFTAFT_1', 'ep_SHIP_DRAFTFOR_1', 'ep_SHIP_HEAD_1', 
    'miglia', 'SHA_POW_TOT', 'SHG_POW_TOT', 'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1', 
    'ep_WH_SWELLP_1', 'ep_WH_SPEED_1', 'ep_WH_DIR_1', 'ep_WH_AIRT_1', 'ep_SHIP_LAT_1', 'ep_SHIP_LON_1'
]

weather_features = [
    'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1', 'ep_WH_SWELLP_1', 
    'ep_WH_SPEED_1', 'ep_WH_DIR_1', 'ep_WH_AIRT_1'
]

beaufort_to_waveh = {0: 0.0, 1: 0.1, 2: 0.2, 3: 0.6, 4: 1.0, 5: 2.0, 6: 3.0, 7: 4.0, 8: 5.5, 9: 7.0}

df_historical = pd.read_csv(r"C:\Users\siani\Desktop\DF\refined_15min_ECO_ADRIATICA.csv")
df_historical['ts'] = pd.to_datetime(df_historical['ts'], format='mixed', dayfirst=True)

simulation_datasets = {}

for leg in schedule_data:
    dep_port = leg['departure_port'].upper()
    arr_port = leg['arrival_port'].upper()
    route_str = f"{dep_port} to {arr_port}"
    route_key = f"{dep_port}_{arr_port}"
    
    target_sog = leg['avg_speed']
    target_beaufort = leg.get('beaufort_level', 1)
    target_waveh = beaufort_to_waveh.get(target_beaufort, 1.0)
    distance = leg['distance_nm']
    
    dep_time_str = leg.get('departure_time', leg.get('departure_dt', leg.get('Departure_Date', '2026-02-01')))
    target_month = pd.to_datetime(dep_time_str).month
    
    draft_aft = leg['draft_aft']
    draft_for = leg['draft_fore']
    
    cruise_hours = distance / target_sog
    n_samples = int(cruise_hours * 4) + 16 
    
    def get_route_season(t_route, t_month):
        df_rs = df_historical[(df_historical['Route'] == t_route) & (df_historical['ts'].dt.month == t_month)].copy()
        if df_rs.empty:
            df_rs = df_historical[(df_historical['Route'] == t_route) & (df_historical['ts'].dt.month.isin([t_month-1, t_month, t_month+1]))].copy()
        if df_rs.empty:
            df_rs = df_historical[df_historical['Route'] == t_route].copy()
        return df_rs
        
    df_pool = get_route_season(route_str, target_month)
        
    if len(df_pool) < 16:
        print(f"[{route_str}] Dati storici insufficienti. Skipped.")
        continue

    voyage_errors = []
    for v in df_pool['Voyage_ID'].unique():
        df_v = df_pool[df_pool['Voyage_ID'] == v].copy()
        if len(df_v) < 10: continue
        
  
        sog_err = abs(df_v['ep_SHIP_SOG_1'].mean() - target_sog) / target_sog
        
 
        wave_mean_err = abs(df_v['ep_WH_WAVEH_1'].mean() - target_waveh) / max(target_waveh, 0.5)
        wave_std_err = df_v['ep_WH_WAVEH_1'].std()
        if pd.isna(wave_std_err): wave_std_err = 0.0
  
        duration_v = (df_v['ts'].max() - df_v['ts'].min()).total_seconds() / 3600
        duration_err = abs(duration_v - cruise_hours) / cruise_hours
        
        if cruise_hours > 20:
            total_err = sog_err + (wave_mean_err * 2.0) + (wave_std_err * 3.0) + (duration_err * 1.5)
        else:
            total_err = sog_err + (wave_mean_err * 2.0) + (wave_std_err * 0.5) + (duration_err * 1.5)
            
        voyage_errors.append((v, total_err))
        
    voyage_errors.sort(key=lambda x: x[1])
    top_5_voyages = [x[0] for x in voyage_errors[:5]]
    
    if not top_5_voyages: 
        continue

    for version_idx, best_v in enumerate(top_5_voyages):
        df_base_trip = df_pool[df_pool['Voyage_ID'] == best_v].sort_values('ts').copy()
        
        xp = np.linspace(0, 1, len(df_base_trip))
        x_new = np.linspace(0, 1, n_samples)
        
        df_sim = pd.DataFrame(index=range(n_samples))
        df_sim['ep_SHIP_DRAFTAFT_1'] = draft_aft
        df_sim['ep_SHIP_DRAFTFOR_1'] = draft_for
        
        real_sog = df_base_trip['ep_SHIP_SOG_1'].interpolate(method='linear').ffill().bfill().values
        sim_sog = np.interp(x_new, xp, real_sog)
        df_sim['ep_SHIP_SOG_1'] = sim_sog - np.mean(sim_sog) + target_sog 
        
        real_stw = df_base_trip['ep_SHIP_STW_1'].interpolate(method='linear').ffill().bfill().values
        sim_stw = np.interp(x_new, xp, real_stw)
        real_slip = sim_sog - sim_stw
        df_sim['ep_SHIP_STW_1'] = df_sim['ep_SHIP_SOG_1'] - real_slip
        df_sim['miglia'] = df_sim['ep_SHIP_SOG_1'] / 4.0

        for col in ['ep_SHIP_HEAD_1', 'ep_SHIP_LAT_1', 'ep_SHIP_LON_1']:
            if col in df_base_trip.columns:
                real_seq = df_base_trip[col].interpolate(method='linear').ffill().bfill().values
                df_sim[col] = np.interp(x_new, xp, real_seq)
            else:
                df_sim[col] = 0.0

        for w_col in weather_features:
            if w_col in df_base_trip.columns:
                real_w = df_base_trip[w_col].interpolate(method='linear').ffill().bfill().values
                df_sim[w_col] = np.interp(x_new, xp, real_w)
            else:
                df_sim[w_col] = 0.0

        if 'SHA_POW_TOT' in df_base_trip.columns:
            real_sha = df_base_trip['SHA_POW_TOT'].interpolate(method='linear').ffill().bfill().values
            sim_sha_interp = np.interp(x_new, xp, real_sha)
            
            v_ratio = df_sim['ep_SHIP_SOG_1'] / np.where(sim_sog < 0.1, 0.1, sim_sog)
            real_draft_mean = (df_base_trip['ep_SHIP_DRAFTAFT_1'].mean() + df_base_trip['ep_SHIP_DRAFTFOR_1'].mean()) / 2.0
            target_draft_mean = (draft_aft + draft_for) / 2.0
            draft_ratio = target_draft_mean / real_draft_mean if real_draft_mean > 0 else 1.0
            
            df_sim['SHA_POW_TOT'] = sim_sha_interp * (v_ratio ** 3) * (draft_ratio ** (2/3))
        else:
            df_sim['SHA_POW_TOT'] = 0.0
            
        if 'SHG_POW_TOT' in df_base_trip.columns:
            real_shg = df_base_trip['SHG_POW_TOT'].interpolate(method='linear').ffill().bfill().values
            df_sim['SHG_POW_TOT'] = np.interp(x_new, xp, real_shg)
        else:
            df_sim['SHG_POW_TOT'] = 0.0
            
        for col in lstm_expected_inputs:
            if col not in df_sim.columns:
                df_sim[col] = 0.0
                
        df_sim = df_sim[lstm_expected_inputs]
        
        dict_key = f"{route_key}_VERSION{version_idx+1}"
        if dict_key not in simulation_datasets: 
            simulation_datasets[dict_key] = []
            
        simulation_datasets[dict_key].append(df_sim)

print("Generation complete")


Generazione dei profili temporali dinamici (Integrità Geo-Fisica pura, Nessun Fallback)...
Generation complete! 5 VERSIONI PURAMENTE FISICHE generate (Senza Fallback).


### Prediction

In [ ]:
print("Loading LSTM Model and Scalers...")
model_path = r"C:\Users\siani\LSTM\model_ECO_ADRIATICA_igwo.keras"
scaler_x_path = r"C:\Users\siani\LSTM\scaler_x_ECO_ADRIATICA.pkl"
scaler_y_path = r"C:\Users\siani\LSTM\scaler_y_ECO_ADRIATICA.pkl"

lstm_model = tf.keras.models.load_model(model_path)
scaler_x = joblib.load(scaler_x_path)
scaler_y = joblib.load(scaler_y_path)
print("Model Loaded Successfully!\n")

results_by_route = {}

for scenario_name, df_sim_list in simulation_datasets.items():
    
    if '_VERSION' not in scenario_name:
        print(f"Skipping incorrectly formatted scenario name: {scenario_name}")
        continue
    
    me1_runs = []
    me2_runs = []
    
    for df_sim in df_sim_list:
        
        # CLIP AI BOUNDS DEL TRAINING SET 
        for i, col in enumerate(df_sim.columns):
            sc_min = scaler_x.data_min_[i]
            sc_max = scaler_x.data_max_[i]
            df_sim[col] = df_sim[col].clip(lower=sc_min, upper=sc_max)

        # SCALING 
        X_scaled = scaler_x.transform(df_sim)
        
        # PADDING
        lookback = 16
        if len(X_scaled) < lookback:
            X_scaled = np.pad(X_scaled, ((lookback - len(X_scaled), 0), (0, 0)), mode='edge')
        
        X_seq = np.array([X_scaled[i : i + lookback] for i in range(len(X_scaled) - lookback + 1)])
        
        # PREDICTION
        pred_scaled = lstm_model.predict(X_seq, verbose=0)
        pred_real = scaler_y.inverse_transform(pred_scaled)
        
        me1_runs.append(np.mean(pred_real[:, 0]))
        me2_runs.append(np.mean(pred_real[:, 1]))
    
   
    avg_me1_flow = np.mean(me1_runs)
    avg_me2_flow = np.mean(me2_runs)
    

    parts = scenario_name.split('_VERSION')
    route_key = parts[0]
    
    if route_key not in results_by_route:
        results_by_route[route_key] = []
        
   
    results_by_route[route_key].append(avg_me1_flow + avg_me2_flow)


# REPORT ENSEMBLED
print("\n" + "="*80)
print("             VOYAGE CONSUMPTION REPORT (CHRONOLOGICAL ENSEMBLE)")
print("="*80)

total_real_week = 0
total_pred_week = 0

for leg in schedule_data:
    dep = leg['departure_port'].upper()
    arr = leg['arrival_port'].upper()
    route_key = f"{dep}_{arr}"
    
    cruise_hours = leg['distance_nm'] / leg['avg_speed']
    real_f = leg.get('real_fuel_mt', 0.0)
    
    if route_key in results_by_route:
        
        # FONDAMENTALE: calcola la MEDIA ARITMETICA di tutte e 5 le versioni stocastiche create
        avg_flow_ensemble = np.mean(results_by_route[route_key])
        
        total_fuel = avg_flow_ensemble * cruise_hours
        err = total_fuel - real_f
        
        icon = "✅" if abs(err) <= 2.0 else "⚠️"
        print(f"{icon} {dep.title():10s} -> {arr.title():10s} | Ensemble | Real: {real_f:4.1f} MT | Pred: {total_fuel:4.1f} MT | Err: {err:+4.1f} MT")
        
        total_real_week += real_f
        total_pred_week += total_fuel

err_tot = total_pred_week - total_real_week
perc = (err_tot / total_real_week) * 100 if total_real_week > 0 else 0


print(f"  TOTAL WEEK | Real: {total_real_week:.1f} MT | Pred: {total_pred_week:.1f} MT | Error: {err_tot:+.1f} MT ({perc:+.2f}%)")



Loading LSTM Model and Scalers...
Model Loaded Successfully!


             VOYAGE CONSUMPTION REPORT (CHRONOLOGICAL ENSEMBLE)
✅ Valencia   -> Barcelona  | Ensemble | Real: 18.2 MT | Pred: 18.6 MT | Err: +0.4 MT
✅ Barcelona  -> Livorno    | Ensemble | Real: 64.5 MT | Pred: 65.5 MT | Err: +1.0 MT
✅ Livorno    -> Savona     | Ensemble | Real: 14.6 MT | Pred: 14.9 MT | Err: +0.3 MT
✅ Savona     -> Barcelona  | Ensemble | Real: 51.7 MT | Pred: 50.7 MT | Err: -1.0 MT
✅ Barcelona  -> Valencia   | Ensemble | Real: 17.6 MT | Pred: 18.3 MT | Err: +0.7 MT
⚠️ Valencia   -> Savona     | Ensemble | Real: 61.2 MT | Pred: 59.1 MT | Err: -2.1 MT
✅ Savona     -> Livorno    | Ensemble | Real: 10.9 MT | Pred:  9.7 MT | Err: -1.2 MT

  TOTAL WEEK | Real: 238.7 MT | Pred: 236.6 MT | Error: -2.1 MT (-0.86%)
